# Taller práctico: Análisis hidroclimático y evaluación del riesgo hídrico en la cuenca del río Bogotá mediante Google Colab

**Escuela ITS2026 · Sesión 3 · Fase presencial, martes 1 de diciembre de 2026**

Un solo ejercicio, de principio a fin, sobre la cuenca del río Bogotá. Cada grupo lo ejecuta, usa lo que necesita según su propuesta, lo ajusta y construye sus propios hallazgos. El caso común sirve para practicar tres decisiones que señalaron los comentarios a las propuestas:

1. **Qué datos hay, qué tan buenos son y cuántos faltan** en cada fuente.
2. **Si los datos representan bien el lugar y el periodo** que se quiere estudiar.
3. **Qué se puede hacer de verdad** con los recursos y el tiempo disponibles.

El marco de referencia es el enfoque de riesgo del IPCC AR6 (riesgo = f(peligro, exposición, vulnerabilidad)), resuelto con **un indicador por componente**.

---

### Cómo usarlo (3 pasos)

1. **Archivo → Guardar una copia en Drive** y trabaje sobre su copia.
2. Suba **`Escuela-ITS2026_datos_ideam.zip`** (está en la carpeta de Drive del taller) al panel de archivos de Colab: ícono de carpeta a la izquierda → *Subir al almacenamiento de la sesión*. En la celda **0.1** escriba el ID de su proyecto de Google Earth Engine y su número de grupo.
3. **Entorno de ejecución → Ejecutar todas**. La primera vez Colab pedirá autorizar el acceso a Earth Engine.

Las celdas marcadas con 🔧 tienen parámetros que puede cambiar para probar su propuesta (sección 8). Las demás no requieren modificación. Todos los resultados (tablas CSV y figuras PNG) quedan en la carpeta `Escuela-ITS2026_resultados` y se descargan al final (9.7).

### Secciones

| Sección | Contenido | Resultado |
|---|---|---|
| 0 · Configuración | Parámetros, datos del taller y conexión con Earth Engine | Entorno listo |
| 1 · Cuenca | Límite, subcuencas, relieve, embalses | Mapa interactivo |
| 2 · Inventario IDEAM | Estaciones en la cuenca y cuántos datos tienen en 1991–2020 | Tabla y mapa de datos disponibles |
| 3 · Precipitación y temperatura | Estación vs. CHIRPS y ERA5-Land; cambio de temperatura con la altura | Gráficos de comparación y °C por km |
| 4 · Variabilidad (ENOS) | Anomalías de lluvia en El Niño, Neutro y La Niña | Barras de anomalía por fase |
| 5 · Extremos y tendencias | Índices de lluvia extrema y prueba de tendencia | Mapa de tendencias |
| 6 · Embalses y lagunas | Área inundada de embalses 1984–hoy | Series de área |
| 7 · Índice de riesgo mínimo | Peligro, exposición y vulnerabilidad por subcuenca | Mapa de riesgo relativo |
| 8 · Su propuesta | Qué partes del notebook usa su grupo, qué cambia y qué datos propios agrega | Registro de lo que usa y cambia |
| 9 · Hallazgos | Tabla de hallazgos, figura propia, lámina, matriz de alcance y ajustes, construidos por el grupo | Lo que el grupo presenta |

### Datos abiertos utilizados

**Estaciones:** series diarias de 10 estaciones de referencia del IDEAM descargadas de DHIME (precipitación y temperatura media, 1991–2025) y Catálogo Nacional de Estaciones, todo en `Escuela-ITS2026_datos_ideam.zip`. **Satélite y reanálisis (Earth Engine):** CHIRPS, ERA5-Land, JRC Global Surface Water, Sentinel-2, HydroBASINS, Copernicus DEM, ESA WorldCover, WorldPop, WDPA, FAO GAUL y el índice ONI de NOAA. Ninguno requiere licencia.

## 0. Configuración

In [ ]:
#@title 0.1 Parámetros principales 🔧
#@markdown Escriba el ID de su proyecto de Google Cloud con Earth Engine habilitado (por ejemplo `ee-minombre`).
PROYECTO_GEE = "mi-proyecto-gee"  #@param {type:"string"}
#@markdown Número de su grupo (0 = sin grupo): ajusta la guía de la sección 8 y el nombre de sus resultados.
GRUPO = 0  #@param [0, 1, 2, 3, 4, 5, 6] {type:"raw"}
#@markdown **Datos del taller:** suba `Escuela-ITS2026_datos_ideam.zip` (o los CSV que contiene) al panel de archivos de Colab (ícono de carpeta, a la izquierda) antes de ejecutar. Si no lo hace, el notebook le mostrará un botón para subirlo.

In [ ]:
# 0.2 Parámetros avanzados 🔧 (para la primera ejecución no es necesario cambiarlos; ver sección 8)

# --- Cuenca --------------------------------------------------------------
CUENCA_NOMBRE = "Río Bogotá"
PUNTO_CUENCA = (-74.635, 4.458)   # (lon, lat) de un punto sobre el río (Tocaima)
AREA_MAX_KM2 = 7000               # el trazado baja por el río mientras el área que drena hacia él sea menor a este valor
NIVEL_HYBAS = 9                   # nivel de HydroBASINS para las subcuencas (8 = menos y más grandes; 10 = más y más pequeñas)

# --- Periodos --------------------------------------------------------------
PERIODO_REF = (1991, 2020)        # periodo de referencia del clima (30 años)
ANIO_INICIO_SAT = 1981            # inicio de CHIRPS
ANIO_FIN = 2025                   # último año a analizar (se ajusta a lo disponible)

# --- Estaciones IDEAM ------------------------------------------------------
CARPETA_DATOS = "/content/Escuela-ITS2026_datos"   # aquí se copian los archivos subidos a la sesión
UMBRAL_DATOS = 80           # % mínimo de días con dato para usar una estación
RELLENAR_CEROS_PPT = True         # DHIME omite muchos días sin lluvia: en meses con registros, los días ausentes = 0 mm

# Estaciones IDEAM de referencia del ejercicio (series descargadas de DHIME, 1991–2025).
# Las estaciones de la cuenca cuyas series de DHIME suba a la sesión se agregan solas a esta lista.
ESTACIONES_REFERENCIA = {
    "21201060": "Pantano Redondo 1 · páramo, cabecera (3.160 m)",
    "21205720": "Granja San Jorge · alta montaña sur (2.900 m)",
    "21205700": "Guasca · zona del embalse de Tominé (2.750 m)",
    "21205740": "Silos · cuenca alta, zona del Sisga (2.709 m)",
    "21205710": "Jardín Botánico · urbana, Bogotá (2.552 m)",
    "21205420": "Tibaitatá · Sabana agrícola (2.543 m)",
    "21205670": "La Florida · cuenca media (1.915 m)",
    "21205660": "Mercedes · cuenca baja (765 m)",
    "21201090": "Viotá · cuenca baja (567 m)",
    "21201100": "Tocaima · cerca de la salida (490 m)",
}

# --- Variabilidad y extremos -----------------------------------------------
INDICE_ENOS = "ONI"               # "ONI" o "RONI"
INDICE_TENDENCIA = "Rx1day"       # PRCPTOT, Rx1day, Rx5day, R95p, R20mm, CDD o TXx
CALCULAR_TXX = True               # TXx a partir de ERA5-Land diario (tarda 1–2 min más)

# --- Embalses y lagunas: nombre -> (lon, lat, radio de búsqueda en m) ---------
EMBALSES = {
    "Tominé":           (-73.831, 4.964, 2500),
    "Neusa":            (-73.967, 5.150, 3000),
    "Sisga":            (-73.717, 5.067, 3000),
    "Laguna de Guatavita": (-73.777, 4.978, 500),
}

# --- Índice de riesgo (un indicador por componente) ------------------------
PELIGRO = "dias_secos"            # "dias_secos" | "rx1day" | "ppt_anual"
EXPOSICION = "poblacion"          # "poblacion" | "cultivos_urbano"
VULNERABILIDAD = "sin_cobertura_natural"  # "sin_cobertura_natural" | "archivo"
PESOS = {"peligro": 1, "exposicion": 1, "vulnerabilidad": 1}
METODO_INDICE = "aritmetico"      # "aritmetico" | "geometrico"
EXPOSICION_LOG = True             # usa log10(1 + densidad) para que la ciudad no domine la escala

# --- Salidas -------------------------------------------------------------
CARPETA_SALIDA = "/content/Escuela-ITS2026_resultados"
REUTILIZAR_DESCARGAS = True       # reutiliza los datos ya descargados en CARPETA_SALIDA
print("Parámetros cargados para la cuenca:", CUENCA_NOMBRE)

In [ ]:
#@title 0.3 Instalación y conexión con Earth Engine (ejecutar)
import sys, subprocess, importlib

def _asegurar(paquete, modulo):
    try:
        importlib.import_module(modulo)
    except ImportError:
        print(f"Instalando {paquete}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

for _p, _m in [("pymannkendall", "pymannkendall"), ("earthengine-api", "ee"),
               ("folium", "folium"), ("geopandas", "geopandas"), ("openpyxl", "openpyxl")]:
    _asegurar(_p, _m)

import ee
if PROYECTO_GEE.strip() in ("", "mi-proyecto-gee"):
    raise ValueError("Escriba el ID de su proyecto de Earth Engine en la celda 0.1 y vuelva a ejecutar.")
try:
    ee.Initialize(project=PROYECTO_GEE)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROYECTO_GEE)
print("Earth Engine conectado con el proyecto:", PROYECTO_GEE)

EN_COLAB = "google.colab" in sys.modules

# Datos del taller: archivos subidos a la sesión (Escuela-ITS2026_datos_ideam.zip, o los CSV / zips de DHIME)
import os, zipfile, shutil as _sh, glob as _glob

def _zips_datos():   # Escuela-ITS2026_datos_ideam.zip (acepta también el nombre anterior datos_ideam*.zip)
    return sorted(_glob.glob("/content/Escuela-ITS2026_datos*.zip") + _glob.glob("/content/datos_ideam*.zip"))

def _hay_datos_subidos():
    return bool(_zips_datos() + _glob.glob("/content/DHIME*.zip") + _glob.glob("/content/*.csv"))

if EN_COLAB and not _hay_datos_subidos():
    from google.colab import files as _files
    print("No encontré los datos del taller en la sesión. Suba Escuela-ITS2026_datos_ideam.zip (o los CSV de DHIME y del catálogo):")
    for _n, _b in _files.upload().items():
        open(os.path.join("/content", _n), "wb").write(_b)
for _z in _zips_datos():
    with zipfile.ZipFile(_z) as z:
        z.extractall("/content/_datos_tmp")
if os.path.isdir("/content/_datos_tmp"):
    os.makedirs(CARPETA_DATOS, exist_ok=True)
    for f in _glob.glob("/content/_datos_tmp/**/*.*", recursive=True):
        os.replace(f, os.path.join(CARPETA_DATOS, os.path.basename(f)))
# CSV sueltos o zips de DHIME subidos directamente a la sesión
for f in _glob.glob("/content/*.csv") + _glob.glob("/content/DHIME*.zip"):
    os.makedirs(CARPETA_DATOS, exist_ok=True)
    _sh.copy(f, os.path.join(CARPETA_DATOS, os.path.basename(f)))
n_arch = len(_glob.glob(os.path.join(CARPETA_DATOS, "*"))) if os.path.isdir(CARPETA_DATOS) else 0
print(f"Archivos de datos del taller en {CARPETA_DATOS}: {n_arch}")

# Último dato disponible de cada producto, para no pedir meses que aún no existen
def _ultimo(coleccion):
    return ee.Date(ee.ImageCollection(coleccion).aggregate_max("system:time_start")).format("YYYY-MM-dd").getInfo()

_ult_chirps = _ultimo("UCSB-CHG/CHIRPS/DAILY")
_ult_era5 = _ultimo("ECMWF/ERA5_LAND/MONTHLY_AGGR")
def _ultimo_anio_completo(fecha):
    a, m = int(fecha[:4]), int(fecha[5:7])
    return a if m == 12 else a - 1
ANIO_FIN_EF = min(ANIO_FIN, _ultimo_anio_completo(_ult_chirps), _ultimo_anio_completo(_ult_era5))
print(f"Último dato CHIRPS: {_ult_chirps} · ERA5-Land: {_ult_era5} · se analiza hasta {ANIO_FIN_EF}")

In [ ]:
#@title 0.4 Funciones de apoyo (ejecutar; no es necesario modificar)
import os, io, re, glob, time, calendar, warnings, zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from shapely.geometry import Point, mapping, shape
import pymannkendall as mk
import folium
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
os.makedirs(CARPETA_SALIDA, exist_ok=True)

# Respaldo: si el paquete de datos trae las series satelitales ya extraídas, se usan y no se repite la consulta
RESPALDO_SATELITAL = ["chirps_diario_puntos", "era5_mensual_puntos", "chirps_mensual_cuenca",
                      "era5_tmax_diaria_puntos", "6_area_cuerpos_agua", "7_indicadores_subcuencas"]
for _n in RESPALDO_SATELITAL:
    _o, _d = os.path.join(CARPETA_DATOS, f"{_n}.csv"), os.path.join(CARPETA_SALIDA, f"{_n}.csv")
    if os.path.exists(_o) and not os.path.exists(_d):
        import shutil
        shutil.copy(_o, _d)
        print(f"Respaldo satelital disponible: {_n}.csv")

CRS_PLANO = "EPSG:3116"   # MAGNA-SIRGAS / Colombia Bogotá, para áreas en km²
COLORES_FASE = {"El Niño": "#d6604d", "Neutro": "#9e9e9e", "La Niña": "#4393c3"}

# ---------------------------------------------------------------- utilidades
def guardar_fig(fig, nombre):
    ruta = os.path.join(CARPETA_SALIDA, f"{nombre}.png")
    fig.savefig(ruta, bbox_inches="tight", dpi=150)
    return ruta

def guardar_tabla(df, nombre, index=False):
    ruta = os.path.join(CARPETA_SALIDA, f"{nombre}.csv")
    df.to_csv(ruta, index=index, encoding="utf-8-sig")
    return ruta

def cache(nombre):
    return os.path.join(CARPETA_SALIDA, f"{nombre}.csv")

def leer_cache(nombre, fechas=("fecha",)):
    ruta = cache(nombre)
    if REUTILIZAR_DESCARGAS and os.path.exists(ruta):
        df = pd.read_csv(ruta, dtype={"codigo": str})
        for c in fechas:
            if c in df:
                df[c] = pd.to_datetime(df[c])
        print(f"  (reutilizando {os.path.basename(ruta)})")
        return df
    return None

def norm_cod(c):
    s = str(c).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s.lstrip("0") or "0"

def a_fecha(serie, dayfirst=False):
    """Convierte fechas en formatos mixtos; las ISO (AAAA-MM-DD) nunca se leen con el día primero."""
    serie = pd.Series(serie)
    if pd.api.types.is_datetime64_any_dtype(serie):
        return serie
    texto = serie.astype(str).str.strip()
    iso = texto.str.match(r"^\d{4}-")
    out = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]")
    for mascara, df_ in [(iso, False), (~iso, dayfirst)]:
        if mascara.any():
            try:
                v = pd.to_datetime(texto[mascara], errors="coerce", format="mixed", dayfirst=df_)
            except (TypeError, ValueError):
                v = pd.to_datetime(texto[mascara], errors="coerce", dayfirst=df_)
            out[mascara] = v.astype("datetime64[ns]")
    return out

def a_numero(serie):
    return pd.to_numeric(serie.astype(str).str.replace(",", ".", regex=False).str.strip(), errors="coerce")

# ---------------------------------------------------------- Earth Engine
def fc_a_gdf(fc):
    feats = fc.getInfo()["features"]
    if not feats:
        return gpd.GeoDataFrame({"geometry": []}, geometry="geometry", crs="EPSG:4326")
    return gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326")

def fc_a_df(fc):
    return pd.DataFrame([f["properties"] for f in fc.getInfo()["features"]])

def puntos_a_fc(df):
    return ee.FeatureCollection([ee.Feature(ee.Geometry.Point([float(r.lon), float(r.lat)]), {"codigo": str(r.codigo)})
                                 for r in df.itertuples()])

def mapa_folium(lat, lon, zoom=9):
    """Mapa base para Colab. OpenStreetMap bloquea sus teselas dentro de Colab y CartoDB pide clave: se usan teselas de Esri."""
    m = folium.Map(location=[lat, lon], zoom_start=zoom, tiles=None)
    folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
                     attr="Esri, HERE, Garmin", name="Mapa claro (Esri)").add_to(m)
    folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
                     attr="Esri, HERE, Garmin, USGS, NGA", name="Topográfico (Esri)", show=False).add_to(m)
    folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                     attr="Esri, Maxar, Earthstar Geographics", name="Imagen satelital (Esri)", show=False).add_to(m)
    return m

def capa_ee(mapa, objeto, vis, nombre, mostrar=True):
    """Agrega una imagen o colección de Earth Engine a un mapa folium."""
    if isinstance(objeto, (ee.Geometry, ee.Feature, ee.FeatureCollection)):
        fc = objeto if isinstance(objeto, ee.FeatureCollection) else ee.FeatureCollection([ee.Feature(objeto)])
        img, vis_img = fc.style(**vis), {}
    else:
        img, vis_img = ee.Image(objeto), vis
    url = img.getMapId(vis_img)["tile_fetcher"].url_format
    folium.raster_layers.TileLayer(tiles=url, attr="Google Earth Engine", name=nombre,
                                   overlay=True, control=True, show=mostrar).add_to(mapa)

def serie_en_puntos(coleccion, bandas, puntos, escala, inicio, fin, dias_trozo=3000, etiqueta=""):
    """Extrae series de una ImageCollection en puntos (codigo, lon, lat) con getRegion, por tramos."""
    pts = puntos[["codigo", "lon", "lat"]].drop_duplicates("codigo").reset_index(drop=True)
    geom = ee.Geometry.MultiPoint(pts[["lon", "lat"]].astype(float).values.tolist())
    t0, t1 = pd.Timestamp(inicio), pd.Timestamp(fin) + pd.Timedelta(days=1)
    partes, a, paso = [], t0, int(dias_trozo)
    while a < t1:
        b = min(a + pd.Timedelta(days=paso), t1)
        col = coleccion.filterDate(a.strftime("%Y-%m-%d"), b.strftime("%Y-%m-%d")).select(list(bandas))
        res = None
        for intento in range(3):
            try:
                res = col.getRegion(geom, escala).getInfo()
                break
            except Exception as e:
                msg = str(e)
                if re.search(r"memory|too many|limit|timed out|payload|too large", msg, re.I) and paso > 31:
                    break   # consulta demasiado grande: se reduce el tramo
                if intento == 2:
                    raise
                print("  reintentando:", msg[:120])
                time.sleep(5)
        if res is None:
            paso = max(31, paso // 2)
            print(f"  {etiqueta}: consulta demasiado grande para Earth Engine; se reduce el tramo a {paso} días")
            continue
        if len(res) > 1:
            partes.append(pd.DataFrame(res[1:], columns=res[0]))
        print(f"  {etiqueta} {a.date()} → {(b - pd.Timedelta(days=1)).date()} ✓")
        a = b
    df = pd.concat(partes, ignore_index=True)
    df["fecha"] = pd.to_datetime(df["time"], unit="ms").dt.normalize()
    df = df.drop_duplicates(["longitude", "latitude", "fecha"])
    pix = df[["longitude", "latitude"]].drop_duplicates().reset_index(drop=True)
    asign = []
    for p in pts.itertuples():
        d = np.hypot(pix["longitude"] - float(p.lon), pix["latitude"] - float(p.lat))
        j = d.idxmin()
        asign.append((p.codigo, pix.at[j, "longitude"], pix.at[j, "latitude"]))
    asign = pd.DataFrame(asign, columns=["codigo", "longitude", "latitude"])
    out = asign.merge(df, on=["longitude", "latitude"], how="left")
    for b in bandas:
        out[b] = pd.to_numeric(out[b], errors="coerce")
    return out[["codigo", "fecha"] + list(bandas)].sort_values(["codigo", "fecha"]).reset_index(drop=True)

def serie_mensual_region(funcion_imagen, geom, anio_ini, anio_fin, escala, nombre):
    """Serie mensual del promedio espacial de una imagen mensual construida por funcion_imagen(d0, d1)."""
    n = (anio_fin - anio_ini + 1) * 12
    def f(m):
        d0 = ee.Date.fromYMD(anio_ini, 1, 1).advance(m, "month")
        d = funcion_imagen(d0, d0.advance(1, "month")).reduceRegion(
            ee.Reducer.mean(), geom, escala, maxPixels=1e10, bestEffort=True)
        v = ee.Algorithms.If(d.size().gt(0), d.values().get(0), None)
        return ee.Feature(None, {"fecha": d0.format("YYYY-MM-dd"), nombre: v})
    df = fc_a_df(ee.FeatureCollection(ee.List.sequence(0, n - 1).map(f)))
    df["fecha"] = pd.to_datetime(df["fecha"])
    df[nombre] = pd.to_numeric(df.get(nombre), errors="coerce")
    return df.set_index("fecha")[nombre]

# ----------------------------------------------------------- cuenca
def delimitar_cuenca(punto, area_max, nivel):
    """Traza la cuenca aguas arriba con HydroBASINS (NEXT_DOWN) a partir de un punto sobre el río."""
    hb = ee.FeatureCollection(f"WWF/HydroSHEDS/v1/Basins/hybas_{nivel}")
    p = ee.Geometry.Point(list(punto))
    unidad = ee.Feature(hb.filterBounds(p).first())
    id0, main_bas = [int(x) for x in ee.List([unidad.get("HYBAS_ID"), unidad.get("MAIN_BAS")]).getInfo()]
    props = None
    for radio_km in (400, 250, 150):
        try:
            tabla = (hb.filter(ee.Filter.eq("MAIN_BAS", main_bas)).filterBounds(p.buffer(radio_km * 1000))
                       .select(["HYBAS_ID", "NEXT_DOWN", "UP_AREA", "SUB_AREA"], None, False))
            props = fc_a_df(tabla)
            break
        except Exception as e:
            print(f"  radio {radio_km} km demasiado grande, reduciendo… ({str(e)[:60]})")
    for c in ["HYBAS_ID", "NEXT_DOWN"]:
        props[c] = props[c].astype("int64")
    info = props.set_index("HYBAS_ID")
    salida = id0
    while True:   # baja por el río mientras el área aportante no supere area_max
        nd = int(info.loc[salida, "NEXT_DOWN"])
        if nd == 0 or nd not in info.index or info.loc[nd, "UP_AREA"] > area_max:
            break
        salida = nd
    hijos = props.groupby("NEXT_DOWN")["HYBAS_ID"].apply(list).to_dict()
    ids, pila = [], [salida]
    while pila:
        u = pila.pop()
        ids.append(int(u))
        pila.extend(hijos.get(u, []))
    fc = hb.filter(ee.Filter.inList("HYBAS_ID", ids))
    gdf = fc_a_gdf(fc)
    gdf["HYBAS_ID"] = gdf["HYBAS_ID"].astype("int64")
    return gdf, fc, float(info.loc[salida, "UP_AREA"])

def poligono_cuerpo_agua(lon, lat, radio):
    """Polígono de máxima extensión de un cuerpo de agua (JRC, ocurrencia > 10 %) cerca de un punto."""
    p = ee.Geometry.Point([lon, lat])
    occ = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence")
    vec = occ.gt(10).selfMask().reduceToVectors(geometry=p.buffer(radio + 8000), scale=30, geometryType="polygon",
                                                eightConnected=True, maxPixels=1e10, bestEffort=True)
    vec = vec.filterBounds(p.buffer(radio)).map(lambda f: f.set("area_max_km2", f.geometry().area(10).divide(1e6)))
    if vec.size().getInfo() == 0:
        return None
    return ee.Feature(vec.sort("area_max_km2", False).first())

# --------------------------------------------------- Catálogo de estaciones
def _col(df, *claves):
    cols = list(df.columns)
    for k in claves:                          # coincidencia exacta primero
        for c in cols:
            if c.lower() == k:
                return c
    for k in claves:
        for c in cols:
            if k in c.lower():
                return c
    return None

def estandarizar_cne(df):
    campos = {"codigo": ("codigo", "código", "codigoestacion"), "nombre": ("nombre", "nombreestacion"),
              "categoria": ("categoria", "categor"), "tecnologia": ("tecnologia", "tecnolog"), "estado": ("estado",),
              "departamento": ("departamento",), "municipio": ("municipio",), "altitud": ("altitud",),
              "fecha_instalacion": ("fecha_instalacion", "instalaci"), "fecha_suspension": ("fecha_suspension", "suspensi"),
              "entidad": ("entidad",), "subzona": ("subzona",), "corriente": ("corriente",)}
    out = pd.DataFrame(index=df.index)
    for k, claves in campos.items():
        c = _col(df, *claves)
        out[k] = df[c] if c is not None else np.nan
    clat, clon = _col(df, "latitud", "lat"), _col(df, "longitud", "lon")
    if clat is not None and clon is not None and not df[clat].apply(lambda v: isinstance(v, dict)).any():
        out["lat"], out["lon"] = a_numero(df[clat]), a_numero(df[clon])
    else:
        cgeo = next((c for c in df.columns if df[c].apply(lambda v: isinstance(v, dict)).any()), None)
        def _ll(v, i):
            if not isinstance(v, dict):
                return np.nan
            if "coordinates" in v:
                return float(v["coordinates"][1 - i])
            return float(v.get("latitude" if i == 0 else "longitude", np.nan))
        out["lat"] = df[cgeo].apply(lambda v: _ll(v, 0)) if cgeo else np.nan
        out["lon"] = df[cgeo].apply(lambda v: _ll(v, 1)) if cgeo else np.nan
    out["codigo"] = out["codigo"].map(norm_cod)
    out["altitud"] = a_numero(out["altitud"])
    for c in ["fecha_instalacion", "fecha_suspension"]:
        out[c] = a_fecha(out[c], dayfirst=True)
    for c in ["nombre", "categoria", "tecnologia", "estado", "departamento", "municipio"]:
        out[c] = out[c].astype(str).str.strip().replace({"nan": np.nan})
    out = out[out["lat"].between(-5, 14) & out["lon"].between(-82, -66)]
    return out.drop_duplicates("codigo").reset_index(drop=True)

def leer_tabla(ruta):
    if ruta.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta)
    for enc in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=enc)
        except Exception:
            continue
    raise ValueError(f"No se pudo leer {ruta}")

def cargar_cne():
    """Catálogo Nacional de Estaciones (archivo CNE_rio_bogota.csv del paquete del taller)."""
    locales = [f for f in glob.glob(os.path.join(CARPETA_DATOS, "*"))
               if re.search(r"cne|catalogo|catálogo", os.path.basename(f), re.I)]
    if not locales:
        raise FileNotFoundError("No encontré CNE_rio_bogota.csv. Suba Escuela-ITS2026_datos_ideam.zip a la sesión y ejecute desde la celda 0.3.")
    cne = estandarizar_cne(leer_tabla(locales[0]))
    print(f"Catálogo leído del archivo {os.path.basename(locales[0])}: {len(cne)} estaciones")
    return cne

# ------------------------------------------------------- series de estaciones
def qc_valores(df):
    lim = {"ppt": (0, 300), "tmed": (-15, 45), "tmax": (-15, 45), "tmin": (-20, 40)}
    for v, (lo, hi) in lim.items():
        m = (df["variable"] == v) & ~df["valor"].between(lo, hi)
        df.loc[m, "valor"] = np.nan
    return df

def variable_parametro(texto):
    """Variable a partir de la descripción del parámetro (formato DHIME sin columna Etiqueta)."""
    t = str(texto).lower()
    if "pluviom" in t or "precipit" in t:
        return "ppt"
    if "temperatura" in t:
        if "máxima" in t or "maxima" in t:
            return "tmax"
        if "mínima" in t or "minima" in t:
            return "tmin"
        return "tmed"
    return None

def rellenar_ceros_ppt(s):
    """DHIME omite muchos días sin lluvia: dentro de cada mes con registros, los días ausentes se toman como 0 mm."""
    p = s[s["variable"] == "ppt"].dropna(subset=["valor"])
    if p.empty:
        return s
    partes = []
    for c, g in p.groupby("codigo"):
        meses = g["fecha"].dt.to_period("M").unique()
        dias = pd.DatetimeIndex(np.concatenate([pd.date_range(m.start_time, m.end_time.normalize()) for m in meses]))
        v = g.set_index("fecha")["valor"].reindex(dias).fillna(0.0)
        partes.append(pd.DataFrame({"codigo": c, "fecha": dias, "variable": "ppt", "valor": v.values}))
    return pd.concat([s[s["variable"] != "ppt"]] + partes, ignore_index=True)

def variable_dhime(etiqueta):
    e = str(etiqueta).upper()
    if e.startswith("PT"):
        return "ppt"
    if "TMX" in e or "TMAX" in e:
        return "tmax"
    if "TMN" in e or "TMIN" in e:
        return "tmin"
    if e.startswith("TS") or "TMED" in e or e.startswith("TA"):
        return "tmed"
    return None

def cargar_archivos_locales(carpeta):
    """Lee las series diarias exportadas de DHIME (CSV o los zip tal como salen del portal)."""
    for z in glob.glob(os.path.join(carpeta, "**", "DHIME*.zip"), recursive=True):   # zips tal como salen de DHIME
        with zipfile.ZipFile(z) as zz:
            for n in zz.namelist():
                if n.lower().endswith(".csv"):
                    destino = os.path.join(carpeta, os.path.splitext(os.path.basename(z))[0] + ".csv")
                    if not os.path.exists(destino):
                        open(destino, "wb").write(zz.read(n))
    archivos = [f for f in glob.glob(os.path.join(carpeta, "**", "*"), recursive=True)
                if f.lower().endswith((".csv", ".txt", ".xlsx", ".xls"))
                and not re.search(r"cne|catalogo|catálogo|vulnerabilidad|lista_estaciones|estaciones_referencia|leeme|chirps|era5|area_cuerpos|indicadores_subcuencas", os.path.basename(f), re.I)]
    series, meta = [], []
    for f in archivos:
        try:
            df = leer_tabla(f)
        except Exception as e:
            print("  omitido", os.path.basename(f), e)
            continue
        cols = {c.lower(): c for c in df.columns}
        if {"codigoestacion", "fecha", "valor"} <= set(cols):           # formato DHIME
            d = pd.DataFrame({"codigo": df[cols["codigoestacion"]].map(norm_cod),
                              "fecha": a_fecha(df[cols["fecha"]], dayfirst=True),
                              "variable": (df[cols["etiqueta"]].map(variable_dhime) if "etiqueta" in cols
                                           else df[cols["parametro"]].map(variable_parametro) if "parametro" in cols else None),
                              "valor": a_numero(df[cols["valor"]])})
            m = {"latitud": "lat", "longitud": "lon", "altitud": "altitud", "nombreestacion": "nombre"}
            mm = pd.DataFrame({"codigo": d["codigo"]})
            for k, v in m.items():
                if k in cols:
                    mm[v] = a_numero(df[cols[k]]) if v != "nombre" else df[cols[k]]
            meta.append(mm.drop_duplicates("codigo"))
        else:
            print("  formato no reconocido:", os.path.basename(f))
            continue
        series.append(d.dropna(subset=["fecha", "variable"]))
        print(f"  leído {os.path.basename(f)}: {len(d):,} registros")
    if not series:
        return None, None
    s = pd.concat(series, ignore_index=True).drop_duplicates(["codigo", "fecha", "variable", "valor"])   # archivos repetidos
    s["fecha"] = s["fecha"].dt.normalize()
    agg = {"ppt": "sum", "tmed": "mean", "tmax": "max", "tmin": "min"}
    s = pd.concat([g.groupby(["codigo", "fecha"], as_index=False)["valor"].agg(agg.get(v, "mean")).assign(variable=v)
                   for v, g in s.groupby("variable")], ignore_index=True)
    meta = pd.concat(meta).drop_duplicates("codigo") if meta else None
    if RELLENAR_CEROS_PPT:
        s = rellenar_ceros_ppt(s)
    return qc_valores(s), meta

# --------------------------------------------------------- análisis de series
def porcentaje_datos(series, variable, ini, fin):
    dias = pd.date_range(f"{ini}-01-01", f"{fin}-12-31")
    s = series[(series["variable"] == variable)].dropna(subset=["valor"])
    if s.empty:
        return pd.DataFrame(columns=["codigo", "primer_dato", "ultimo_dato", "dias_validos", "anios_utiles", "pct_datos"])
    s = s.assign(fecha=pd.to_datetime(s["fecha"]))
    total = s.groupby("codigo")["fecha"].agg(primer_dato="min", ultimo_dato="max")
    p = s[s["fecha"].between(dias[0], dias[-1])]
    res = pd.DataFrame({"dias_validos": p.groupby("codigo")["fecha"].nunique()})
    anual = p.groupby(["codigo", p["fecha"].dt.year])["fecha"].nunique()
    res["anios_utiles"] = (anual >= 0.8 * 365).groupby(level=0).sum()
    res = total.join(res, how="left").fillna({"dias_validos": 0, "anios_utiles": 0})
    res["pct_datos"] = (res["dias_validos"] / len(dias) * 100).round(1)
    return res.reset_index()

def mensual(df, col, como="sum", min_frac=0.9):
    """Agrega una serie diaria (codigo, fecha, col) a mensual, solo meses con ≥ min_frac de días."""
    d = df.dropna(subset=[col]).copy()
    if d.empty:
        return pd.DataFrame(columns=["codigo", col, "fecha"])
    d["fecha"] = pd.to_datetime(d["fecha"])
    d["mes"] = d["fecha"].dt.to_period("M")
    g = d.groupby(["codigo", "mes"])[col]
    agg = g.sum() if como == "sum" else g.mean()
    n = g.count()
    dias = np.array([p.days_in_month for p in agg.index.get_level_values("mes")])
    agg = agg[n.values >= min_frac * dias]
    out = agg.reset_index()
    out["fecha"] = out["mes"].dt.to_timestamp()
    return out.drop(columns="mes")

def metricas(obs, sim):
    m = pd.concat([obs, sim], axis=1).dropna()
    o, s = m.iloc[:, 0], m.iloc[:, 1]
    if len(m) < 12 or o.std() == 0:
        return {"n_meses": len(m), "r": np.nan, "sesgo_pct": np.nan, "sesgo_abs": np.nan, "rmse": np.nan, "KGE": np.nan}
    r = np.corrcoef(o, s)[0, 1]
    alfa, beta = s.std() / o.std(), s.mean() / o.mean()
    return {"n_meses": len(m), "r": round(float(r), 2), "sesgo_pct": round(float(beta - 1) * 100, 1),
            "sesgo_abs": round(float(s.mean() - o.mean()), 2), "rmse": round(float(np.sqrt(((s - o) ** 2).mean())), 2),
            "KGE": round(float(1 - np.sqrt((r - 1) ** 2 + (alfa - 1) ** 2 + (beta - 1) ** 2)), 2)}

def indices_etccdi(p, p95=None, min_frac=0.9):
    """Índices ETCCDI anuales de precipitación a partir de una serie diaria (mm) indexada por fecha."""
    p = p.sort_index()
    filas = []
    for anio, g in p.groupby(p.index.year):
        dias = 366 if calendar.isleap(anio) else 365
        if g.notna().sum() < min_frac * dias:
            continue
        seco = ((g < 1) & g.notna()).astype(int)
        cdd = seco.groupby((seco != seco.shift()).cumsum()).sum().max()
        filas.append({"anio": anio, "PRCPTOT": g[g >= 1].sum(), "Rx1day": g.max(),
                      "Rx5day": g.rolling(5, min_periods=5).sum().max(),
                      "R95p": g[g > p95].sum() if p95 is not None else np.nan,
                      "R20mm": int((g >= 20).sum()), "CDD": int(cdd)})
    return pd.DataFrame(filas).set_index("anio") if filas else pd.DataFrame()

def tendencia(serie):
    s = serie.dropna()
    if len(s) < 10:
        return {"n_anios": len(s), "tendencia": "datos insuficientes", "p": np.nan, "sen_por_decada": np.nan}
    r = mk.original_test(s.values)
    t = {"increasing": "aumento", "decreasing": "disminución"}.get(r.trend, "sin tendencia")
    return {"n_anios": len(s), "tendencia": t, "p": round(r.p, 3), "sen_por_decada": round(r.slope * 10, 3)}

# ------------------------------------------------------------------- ENOS
def cargar_indice_enos(nombre="ONI"):
    urls = {"ONI": ["https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt",
                    "https://psl.noaa.gov/data/correlation/oni.data"],
            "RONI": ["https://www.cpc.ncep.noaa.gov/data/indices/RONI.ascii.txt"]}[nombre]
    estaciones = {"DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4, "AMJ": 5, "MJJ": 6,
                  "JJA": 7, "JAS": 8, "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12}
    for url in urls:
        try:
            txt = requests.get(url, timeout=60).text
        except Exception:
            continue
        if "SEAS" in txt[:300]:
            df = pd.read_csv(io.StringIO(txt), sep=r"\s+")
            col = "ANOM" if "ANOM" in df else df.columns[-1]
            idx = pd.to_datetime(dict(year=df["YR"], month=df["SEAS"].map(estaciones), day=1))
            s = pd.Series(pd.to_numeric(df[col], errors="coerce").values, index=idx)
        else:
            filas = [l.split() for l in txt.splitlines()]
            filas = [f for f in filas if len(f) == 13 and f[0].isdigit() and len(f[0]) == 4]
            reg = [(pd.Timestamp(int(f[0]), m + 1, 1), float(f[m + 1])) for f in filas for m in range(12)]
            s = pd.Series(dict(reg)).replace(-99.9, np.nan)
        s = s.dropna().sort_index()
        if len(s) > 100:
            print(f"Índice {nombre} cargado desde {url.split('/')[2]}: {s.index[0]:%Y-%m} a {s.index[-1]:%Y-%m}")
            return s
    raise RuntimeError(f"No se pudo descargar el índice {nombre}.")

def fases_enos(indice, umbral=0.5, n=5):
    """Fase por mes: El Niño / La Niña si el índice supera ±umbral durante ≥ n trimestres seguidos."""
    s = indice.sort_index()
    fase = pd.Series("Neutro", index=s.index)
    for signo, nombre in [(1, "El Niño"), (-1, "La Niña")]:
        cond = (s * signo) >= umbral
        largo = cond.groupby((cond != cond.shift()).cumsum()).transform("size")
        fase[cond & (largo >= n)] = nombre
    return fase

# ---------------------------------------------------------------- mapas
def mapa_base(ax, titulo=""):
    gdf_sub.boundary.plot(ax=ax, color="#9e9e9e", lw=0.5)
    cuenca_gdf.boundary.plot(ax=ax, color="black", lw=1.2)
    emb = globals().get("gdf_emb")
    if emb is not None and len(emb):
        emb.plot(ax=ax, color="#4fc3f7", edgecolor="#0277bd", lw=0.4)
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")
    ax.set_aspect("equal")

print("Funciones listas ✓")

---
## 1. Preparación y cuenca

**Conceptos clave:** la cuenca como área de estudio · nivel de detalle de los datos · mapas de imagen (cuadrícula de celdas) y mapas de límites (líneas y áreas).

La cuenca se traza con **HydroBASINS**: cada unidad sabe hacia cuál otra le corre el agua (`NEXT_DOWN`), así que basta con partir de un punto sobre el río y seguir la red río arriba. Las unidades de nivel 9 se usan como **subcuencas**, que serán la unidad del índice de riesgo en la sección 7. Los embalses se dibujan con la mayor área de agua que han registrado los satélites Landsat (JRC Global Surface Water).

In [ ]:
# 1.1 Límite de la cuenca y subcuencas
print("Trazando el límite de la cuenca con HydroBASINS…")
gdf_sub, fc_sub, area_aportante = delimitar_cuenca(PUNTO_CUENCA, AREA_MAX_KM2, NIVEL_HYBAS)
gdf_sub = gdf_sub.sort_values("UP_AREA", ascending=False).reset_index(drop=True)
gdf_sub["subcuenca"] = [f"SC{i + 1:02d}" for i in range(len(gdf_sub))]
gdf_sub["area_km2"] = (gdf_sub.to_crs(CRS_PLANO).area / 1e6).round(1)
cuenca_gdf = gdf_sub[["geometry"]].dissolve()
geom_cuenca = fc_sub.union(100).geometry()

# Relieve por subcuenca (Copernicus DEM 30 m)
_glo30 = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM")
dem = _glo30.mosaic().setDefaultProjection(_glo30.first().projection()).rename("elev")
reductor = ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True)
relieve = fc_a_df(dem.reduceRegions(fc_sub.select(["HYBAS_ID"]), reductor, 90)
                  .select(["HYBAS_ID", "mean", "min", "max"], None, False))
relieve.columns = ["HYBAS_ID", "elev_media", "elev_min", "elev_max"]
relieve["HYBAS_ID"] = relieve["HYBAS_ID"].astype("int64")
gdf_sub = gdf_sub.merge(relieve.round(0), on="HYBAS_ID", how="left")

print(f"Cuenca {CUENCA_NOMBRE}: {gdf_sub['area_km2'].sum():,.0f} km² en {len(gdf_sub)} subcuencas "
      f"(área que drena según HydroBASINS {area_aportante:,.0f} km²)")
print(f"Altitud: {gdf_sub['elev_min'].min():,.0f} – {gdf_sub['elev_max'].max():,.0f} m")
tabla_sub = gdf_sub[["subcuenca", "HYBAS_ID", "area_km2", "elev_media", "elev_min", "elev_max"]]
guardar_tabla(tabla_sub, "1_subcuencas")
display(tabla_sub)

In [ ]:
# 1.2 Embalses y lagunas (máxima extensión de agua, JRC 1984–2021)
filas, geom_emb = [], {}
for nombre, (lon, lat, radio) in EMBALSES.items():
    f = poligono_cuerpo_agua(lon, lat, radio)
    if f is None:
        print(f"⚠️ No se encontró agua cerca de {nombre}; revise sus coordenadas en EMBALSES.")
        continue
    info = f.getInfo()
    geom_emb[nombre] = f.geometry()
    filas.append({"nombre": nombre, "area_max_km2": round(info["properties"]["area_max_km2"], 2),
                  "geometry": shape(info["geometry"])})
gdf_emb = gpd.GeoDataFrame(filas, geometry="geometry", crs="EPSG:4326")
display(gdf_emb[["nombre", "area_max_km2"]])

In [ ]:
# 1.3 Mapa interactivo (use el control de capas de la esquina superior derecha)
centro = cuenca_gdf.geometry.iloc[0].centroid
m = mapa_folium(centro.y, centro.x)
capa_ee(m, dem.clip(geom_cuenca), {"min": 250, "max": 3800,
        "palette": ["#1a9850", "#a6d96a", "#fee08b", "#f46d43", "#a50026", "#f7f7f7"]}, "Relieve (Copernicus DEM)")
capa_ee(m, ee.Terrain.hillshade(dem).clip(geom_cuenca), {"min": 0, "max": 255, "opacity": 0.4}, "Relieve sombreado", False)
capa_ee(m, ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").selfMask().clip(geom_cuenca),
        {"min": 0, "max": 100, "palette": ["#c6dbef", "#08306b"]}, "Frecuencia de agua (JRC)", False)
folium.GeoJson(gdf_sub[["subcuenca", "area_km2", "elev_media", "geometry"]], name="Subcuencas",
               style_function=lambda f: {"color": "#424242", "weight": 1, "fillOpacity": 0},
               tooltip=folium.GeoJsonTooltip(["subcuenca", "area_km2", "elev_media"],
                                             aliases=["Subcuenca", "Área (km²)", "Altitud media (m)"])).add_to(m)
folium.GeoJson(cuenca_gdf, name="Cuenca", style_function=lambda f: {"color": "black", "weight": 3, "fillOpacity": 0}).add_to(m)
if len(gdf_emb):
    folium.GeoJson(gdf_emb, name="Embalses", style_function=lambda f: {"color": "#0277bd", "fillColor": "#4fc3f7", "weight": 1, "fillOpacity": 0.8},
                   tooltip=folium.GeoJsonTooltip(["nombre", "area_max_km2"], aliases=["Cuerpo de agua", "Área máx. (km²)"])).add_to(m)
folium.LayerControl().add_to(m)
m.save(os.path.join(CARPETA_SALIDA, "1_mapa_cuenca.html"))
m

> **Para interpretar:** ¿cuál es el área de estudio de su propuesta (cuenca, subcuenca, municipio, cuerpo de agua)? ¿Corresponde con el nivel de detalle de los datos que piensa usar (estaciones en un punto, cuadrículas de 5–10 km, imágenes de 10–30 m)?

---
## 2. Inventario IDEAM

**Conceptos clave:** periodo de referencia del clima (30 años) · datos que faltan · cómo escoger estaciones.

Primero se consulta el **Catálogo Nacional de Estaciones (CNE)** para saber qué estaciones existen dentro de la cuenca. Luego se revisa **cuántos datos hay realmente** en la fuente disponible para el periodo de referencia 1991–2020.

**Fuente de los datos:** el catálogo de estaciones de la subzona río Bogotá (datos.gov.co) y las series diarias de **10 estaciones de referencia del IDEAM** descargadas de **DHIME** para 1991–2025. Son 10 series de precipitación (PTPM_CON) y 7 de temperatura media (TSSM_MEDIA_D), elegidas para cubrir alturas de 3.160 a 490 m:

| Estación | Altitud | Papel en el análisis | Variables |
|---|---|---|---|
| Pantano Redondo 1 | 3.160 m | Páramo, cabecera | P |
| Granja San Jorge | 2.900 m | Alta montaña sur | P, T |
| Guasca | 2.750 m | Zona del embalse de Tominé | P, T |
| Silos | 2.709 m | Cuenca alta, zona del Sisga | P, T |
| Jardín Botánico | 2.552 m | Urbana | P, T |
| Tibaitatá | 2.543 m | Sabana agrícola | P, T |
| La Florida | 1.915 m | Cuenca media | P, T |
| Mercedes | 765 m | Cuenca baja | P, T |
| Viotá | 567 m | Cuenca baja | P |
| Tocaima | 490 m | Cerca de la salida | P |

Estas 10 estaciones son los puntos de análisis de las secciones 3 a 5. Si sube a la sesión las series de DHIME de otras estaciones dentro de la cuenca (mismo formato), también entran como puntos de análisis. Los datos vienen en `Escuela-ITS2026_datos_ideam.zip`, que se sube a la sesión de Colab antes de ejecutar. **Respaldo con CHIRPS:** si a una estación le falta su serie de DHIME (o no se subieron las series), el análisis usa CHIRPS en la misma ubicación y lo indica en la salida.

El catálogo incluye estaciones del IDEAM y de otras entidades (CAR, EAAB, IDIGER, entre otras); las series de DHIME corresponden a las del IDEAM.

**Ojo con DHIME:** en muchas estaciones convencionales los días sin lluvia no aparecen en el archivo. Si se cuentan como vacíos, el porcentaje de datos sale más bajo de lo real. Con `RELLENAR_CEROS_PPT = True`, dentro de cada mes que tiene registros los días ausentes se toman como 0 mm, y un mes sin ningún registro sigue contando como vacío. Compare el resultado con `False` para ver la diferencia.

La diferencia entre los **datos esperados** (según las fechas en que la estación funcionó, de acuerdo con el catálogo) y los **datos reales** (días con dato en el archivo) es justo lo que hay que revisar antes de prometer una variable en una propuesta.

In [ ]:
# 2.1 Estaciones del Catálogo Nacional dentro de la cuenca
cne = cargar_cne()
pts = gpd.GeoDataFrame(cne, geometry=gpd.points_from_xy(cne["lon"], cne["lat"]), crs="EPSG:4326")
est = gpd.sjoin(pts, gdf_sub[["subcuenca", "geometry"]], predicate="within", how="inner").drop(columns="index_right")
faltan = pts[pts["codigo"].isin(ESTACIONES_REFERENCIA) & ~pts["codigo"].isin(est["codigo"])]
if len(faltan):   # estaciones de referencia en el borde del polígono de HydroBASINS
    faltan = gpd.sjoin_nearest(faltan.to_crs(CRS_PLANO), gdf_sub[["subcuenca", "geometry"]].to_crs(CRS_PLANO)
                               ).drop(columns="index_right").to_crs("EPSG:4326")
    est = pd.concat([est, faltan], ignore_index=True)

def preparar_inventario(est):
    est = est.copy()
    cat = est["categoria"].fillna("").str.lower()
    est["mide_ppt"] = ~cat.str.contains("limni|mareo|sedim|ambiental|radio")
    est["mide_temp"] = est["mide_ppt"] & ~cat.str.contains("pluvio")
    ini, fin = pd.Timestamp(f"{PERIODO_REF[0]}-01-01"), pd.Timestamp(f"{PERIODO_REF[1]}-12-31")
    a = pd.to_datetime(est["fecha_instalacion"]).fillna(ini)
    a = a.where(a > ini, ini)
    b = pd.to_datetime(est["fecha_suspension"]).fillna(fin)
    b = b.where(b < fin, fin)
    est["datos_esperados_pct"] = ((b - a).dt.days.clip(lower=0) / (fin - ini).days * 100).round(0)
    return est.reset_index(drop=True)

est = preparar_inventario(est)
est["referencia"] = est["codigo"].isin(ESTACIONES_REFERENCIA)
print(f"{len(est)} estaciones del catálogo dentro de la cuenca; {est['referencia'].sum()} de ellas son las estaciones de referencia")
if len(est):
    display(pd.crosstab(est["categoria"], est["estado"], margins=True, margins_name="Total"))
    display(est["entidad"].value_counts().rename("estaciones").to_frame())
    display(est["tecnologia"].value_counts().rename("estaciones").to_frame())

In [ ]:
# 2.2 Series diarias de DHIME (precipitación y temperatura media de las estaciones de referencia)
def obtener_series():
    _c = cache("series_estaciones")
    _arch = glob.glob(os.path.join(CARPETA_DATOS, "*")) if os.path.isdir(CARPETA_DATOS) else []
    if os.path.exists(_c) and _arch and max(map(os.path.getmtime, _arch)) > os.path.getmtime(_c):
        os.remove(_c)   # hay archivos nuevos en la sesión: se vuelven a leer
    s = leer_cache("series_estaciones")
    meta = leer_cache("meta_estaciones", fechas=())
    if s is not None:
        return s, meta
    print(f"Leyendo las series de DHIME en {CARPETA_DATOS}…")
    s, meta = cargar_archivos_locales(CARPETA_DATOS) if os.path.isdir(CARPETA_DATOS) else (None, None)
    if s is None or s.empty:
        print("⚠️ No encontré las series de DHIME: las secciones 3 a 5 usarán CHIRPS como respaldo en las estaciones de referencia."
              " Para usar las series, suba Escuela-ITS2026_datos_ideam.zip y ejecute de nuevo desde la celda 0.3.")
        return pd.DataFrame(columns=["codigo", "fecha", "variable", "valor"]), None
    guardar_tabla(s, "series_estaciones")
    if meta is not None:
        guardar_tabla(meta, "meta_estaciones")
    return s, meta

series, meta = obtener_series()
series["codigo"] = series["codigo"].astype(str)

# Estaciones que vienen en los archivos pero no en el catálogo
if meta is not None and {"lat", "lon"} <= set(meta.columns):
    nuevas = meta[~meta["codigo"].astype(str).isin(est["codigo"])].dropna(subset=["lat", "lon"])
    if len(nuevas):
        g = gpd.GeoDataFrame(nuevas, geometry=gpd.points_from_xy(nuevas["lon"], nuevas["lat"]), crs="EPSG:4326")
        g = gpd.sjoin(g, gdf_sub[["subcuenca", "geometry"]], predicate="within").drop(columns="index_right")
        g["categoria"], g["estado"] = "Sin catálogo", "Desconocido"
        est = preparar_inventario(pd.concat([est, g], ignore_index=True))
        est["mide_temp"] = True

# Puntos de análisis: estaciones de referencia + estaciones de la cuenca con series subidas a la sesión
est["referencia"] = est["codigo"].isin(ESTACIONES_REFERENCIA) | est["codigo"].isin(series["codigo"].unique())
_extra = est.loc[est["referencia"] & ~est["codigo"].isin(ESTACIONES_REFERENCIA), "codigo"].tolist()
if _extra:
    print(f"Estaciones agregadas por tener series subidas: {', '.join(_extra)}")
print(f"Series disponibles: {series['codigo'].nunique()} estaciones, "
      f"{series['fecha'].min():%Y-%m-%d} a {series['fecha'].max():%Y-%m-%d}" if len(series) else "Series disponibles: ninguna")

In [ ]:
# 2.3 Porcentaje de días con dato en el periodo de referencia y selección de estaciones
comp_p = porcentaje_datos(series, "ppt", *PERIODO_REF).rename(columns={
    "pct_datos": "pct_ppt", "anios_utiles": "anios_utiles_ppt", "primer_dato": "primer_dato_ppt", "ultimo_dato": "ultimo_dato_ppt"})
comp_t = porcentaje_datos(series, "tmed", *PERIODO_REF)[["codigo", "pct_datos"]].rename(columns={"pct_datos": "pct_tmed"})
inv = est.merge(comp_p.drop(columns="dias_validos"), on="codigo", how="left").merge(comp_t, on="codigo", how="left")
inv[["pct_ppt", "pct_tmed", "anios_utiles_ppt"]] = inv[["pct_ppt", "pct_tmed", "anios_utiles_ppt"]].fillna(0)
inv["seleccionada"] = inv["pct_ppt"] >= UMBRAL_DATOS

cols = ["codigo", "nombre", "entidad", "categoria", "tecnologia", "estado", "subcuenca", "altitud", "datos_esperados_pct",
        "pct_ppt", "anios_utiles_ppt", "primer_dato_ppt", "ultimo_dato_ppt", "pct_tmed", "seleccionada"]
tabla_inv = inv[cols].sort_values(["pct_ppt", "datos_esperados_pct"], ascending=False)
guardar_tabla(tabla_inv, "2_inventario_datos")

n_met = int(inv["mide_ppt"].sum())
n_dat = int((inv["pct_ppt"] > 0).sum())
n_ok = int(inv["seleccionada"].sum())
print(f"Estaciones meteorológicas en la cuenca: {n_met}")
print(f"Con datos de precipitación en {PERIODO_REF[0]}–{PERIODO_REF[1]} en la fuente consultada: {n_dat}")
print(f"Con al menos {UMBRAL_DATOS} % de días con dato: {n_ok}")
print("\nPuntos de análisis (estaciones de referencia y series subidas):")
ref = inv[inv["referencia"]].assign(papel=lambda d: d["codigo"].map(ESTACIONES_REFERENCIA).fillna("serie subida a la sesión"))
display(ref[["codigo", "papel", "categoria", "subcuenca", "altitud", "pct_ppt", "anios_utiles_ppt", "pct_tmed", "seleccionada"]]
        .sort_values("altitud", ascending=False))
print("\nOtras estaciones del catálogo (sin series descargadas):")
display(tabla_inv[~tabla_inv["codigo"].isin(ref["codigo"])].head(15))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, col, titulo in [(axes[0], "datos_esperados_pct", "Datos esperados según el catálogo (%)"),
                        (axes[1], "pct_ppt", f"Días con dato de lluvia (%) {PERIODO_REF[0]}–{PERIODO_REF[1]} (%)")]:
    mapa_base(ax, titulo)
    d = inv[inv["mide_ppt"]]
    if len(d):
        sc = ax.scatter(d["lon"], d["lat"], c=d[col], cmap="RdYlGn", vmin=0, vmax=100, s=40,
                        edgecolor=np.where(d["seleccionada"], "black", "none"), linewidth=1.2, zorder=3)
        plt.colorbar(sc, ax=ax, shrink=0.7, label="%")
        for r in d[d["referencia"]].itertuples():
            ax.annotate(str(r.nombre).split("[")[0].strip().title(), (r.lon, r.lat), fontsize=6, xytext=(3, 3), textcoords="offset points")
fig.suptitle(f"Estaciones en la cuenca del {CUENCA_NOMBRE} (borde negro = al menos {UMBRAL_DATOS} % de días con dato; con nombre = referencia)")
plt.tight_layout()
guardar_fig(fig, "2_mapa_datos")
plt.show()

> **Para interpretar (grupos 1, 2 y 5):** ¿cuántas de las estaciones del catálogo tienen realmente datos para su periodo? ¿Qué pasa con la selección si cambia `UMBRAL_DATOS` a 70 % o 90 %? ¿Qué entidad produce cada dato que su propuesta necesita y cuánto tarda en obtenerse?

---
## 3. Precipitación y temperatura

**Conceptos clave:** ¿el dato representa el lugar? · comparación con estaciones (r, KGE) y sesgo (diferencia promedio) · reanálisis (modelo del clima ajustado con observaciones) · cambio de la temperatura con la altura.

- **CHIRPS** (≈ 5 km, diario desde 1981) combina satélite y estaciones. Se compara con la precipitación mensual de cada estación.
- **ERA5-Land** (≈ 9 km, reanálisis) se compara con la temperatura de las estaciones. Cada cuadro de 9 km promedia montañas y valles: si la estación está más alta o más baja que el promedio del cuadro, aparece una **diferencia por altura (sesgo)**.
- **KGE** (índice de Kling-Gupta) resume en un número qué tan parecidas son dos series: 1 es igual; por encima de 0,5 se suele considerar aceptable para datos mensuales.

In [ ]:
# 3.1 Puntos de análisis y extracción de CHIRPS y ERA5-Land (tarda 1–3 min la primera vez)
puntos = inv[inv["referencia"]].copy()   # estaciones de referencia + estaciones con series subidas
if puntos.empty:
    raise ValueError("Ninguna estación de ESTACIONES_REFERENCIA aparece en el catálogo; revise los códigos en la celda 0.2.")
puntos = puntos[["codigo", "nombre", "lon", "lat", "altitud", "subcuenca"]].reset_index(drop=True)
puntos["codigo"] = puntos["codigo"].astype(str)
print(f"{len(puntos)} puntos de análisis")

chirps_d = leer_cache("chirps_diario_puntos")
if chirps_d is None or set(puntos["codigo"]) - set(chirps_d["codigo"]):
    chirps_d = serie_en_puntos(ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY"), ["precipitation"], puntos, 5566,
                               f"{ANIO_INICIO_SAT}-01-01", f"{ANIO_FIN_EF}-12-31", etiqueta="CHIRPS")
    chirps_d = chirps_d.rename(columns={"precipitation": "ppt"})
    guardar_tabla(chirps_d, "chirps_diario_puntos")

era_m = leer_cache("era5_mensual_puntos")
if era_m is None or set(puntos["codigo"]) - set(era_m["codigo"]):
    era_m = serie_en_puntos(ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR"),
                            ["temperature_2m", "total_precipitation_sum"], puntos, 11132,
                            f"{ANIO_INICIO_SAT}-01-01", f"{ANIO_FIN_EF}-12-31", dias_trozo=20000, etiqueta="ERA5-Land")
    era_m["t_era5"] = era_m["temperature_2m"] - 273.15
    era_m["ppt_era5"] = era_m["total_precipitation_sum"] * 1000
    era_m = era_m[["codigo", "fecha", "t_era5", "ppt_era5"]]
    guardar_tabla(era_m, "era5_mensual_puntos")

# Altitud del punto (DEM 30 m) y altitud media de la celda de ERA5-Land (≈ 0,1°)
fc_pts = puntos_a_fc(puntos)
alt = fc_a_df(dem.reduceRegions(fc_pts, ee.Reducer.first(), 30).select(["codigo", "first"], None, False))
celdas = ee.FeatureCollection([ee.Feature(ee.Geometry.Rectangle([round(r.lon, 1) - 0.05, round(r.lat, 1) - 0.05,
                                                                round(r.lon, 1) + 0.05, round(r.lat, 1) + 0.05]),
                                          {"codigo": r.codigo}) for r in puntos.itertuples()])
alt_c = fc_a_df(dem.reduceRegions(celdas, ee.Reducer.mean(), 90).select(["codigo", "mean"], None, False))
puntos = (puntos.drop(columns=[c for c in ["alt_dem", "alt_celda_era5"] if c in puntos])
          .merge(alt.rename(columns={"first": "alt_dem"}), on="codigo", how="left")
          .merge(alt_c.rename(columns={"mean": "alt_celda_era5"}), on="codigo", how="left"))
puntos["altitud"] = puntos["altitud"].fillna(puntos["alt_dem"])
puntos["dif_alt_catalogo_dem"] = (puntos["altitud"] - puntos["alt_dem"]).round(0)
display(puntos[["codigo", "nombre", "altitud", "alt_dem", "dif_alt_catalogo_dem", "alt_celda_era5"]].round(0).head(15))
print("Diferencias grandes entre la altitud del catálogo y la del DEM pueden indicar errores de coordenadas.")

In [ ]:
# 3.2 Precipitación: estación vs. CHIRPS (meses con ≥ 90 % de días con dato)
chirps_m = mensual(chirps_d, "ppt", "sum", 1.0).rename(columns={"ppt": "ppt_chirps"})
est_m = mensual(series[series["variable"] == "ppt"].rename(columns={"valor": "ppt"}), "ppt", "sum", 0.9)
comp = est_m.merge(chirps_m, on=["codigo", "fecha"]).merge(puntos[["codigo", "nombre", "altitud"]], on="codigo")
obs_nombre, sim_nombre, x, y = "Estación IDEAM", "CHIRPS", "ppt", "ppt_chirps"
sin_serie = sorted(set(puntos["codigo"]) - set(comp["codigo"]))
if sin_serie:
    print(f"⚠️ Estaciones sin serie de DHIME (se usa CHIRPS como respaldo en las secciones 4 y 5): {', '.join(sin_serie)}")
if comp.empty:   # respaldo total: sin series de estación se compara CHIRPS con ERA5-Land para ilustrar las métricas
    comp = chirps_m.merge(era_m, on=["codigo", "fecha"]).merge(puntos[["codigo", "nombre", "altitud"]], on="codigo")
    obs_nombre, sim_nombre, x, y = "ERA5-Land", "CHIRPS", "ppt_era5", "ppt_chirps"

val_ppt = pd.DataFrame([{"codigo": c, "nombre": g["nombre"].iloc[0], "altitud": g["altitud"].iloc[0],
                         **metricas(g.set_index("fecha")[x], g.set_index("fecha")[y])}
                        for c, g in comp.groupby("codigo")]).sort_values("KGE", ascending=False)
guardar_tabla(val_ppt, "3_validacion_precipitacion")
global_ppt = metricas(comp[x].reset_index(drop=True), comp[y].reset_index(drop=True))
print(f"{obs_nombre} vs. {sim_nombre} (todas las estaciones): {global_ppt}")
display(val_ppt)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.scatter(comp[x], comp[y], s=6, alpha=0.35, color="#1f78b4")
lim = np.nanpercentile(comp[[x, y]].values, 99.5) if len(comp) else 1
ax.plot([0, lim], [0, lim], "k--", lw=1, label="1:1")
ax.set(xlim=(0, lim), ylim=(0, lim), xlabel=f"{obs_nombre} (mm/mes)", ylabel=f"{sim_nombre} (mm/mes)",
       title=f"Precipitación mensual · r = {global_ppt['r']} · KGE = {global_ppt['KGE']}")
ax.legend()
ax = axes[1]
ax.scatter(val_ppt["altitud"], val_ppt["sesgo_pct"], c=val_ppt["KGE"], cmap="RdYlGn", vmin=-0.5, vmax=1, s=45, edgecolor="k")
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel="Altitud de la estación (m)", ylabel=f"Diferencia de {sim_nombre} frente a la estación (%)", title="¿Cambia la diferencia con la altura?")
plt.tight_layout()
guardar_fig(fig, "3_validacion_precipitacion")
plt.show()

In [ ]:
# 3.3 Temperatura: estación vs. ERA5-Land y cambio de la temperatura con la altura
t_est = mensual(series[series["variable"] == "tmed"].rename(columns={"valor": "t"}), "t", "mean", 0.8)
comp_t = t_est.merge(era_m, on=["codigo", "fecha"]).merge(puntos[["codigo", "nombre", "altitud", "alt_celda_era5"]], on="codigo")
val_t = pd.DataFrame([{"codigo": c, "nombre": g["nombre"].iloc[0], "altitud": g["altitud"].iloc[0],
                       "alt_celda_era5": g["alt_celda_era5"].iloc[0], "t_media_est": round(g["t"].mean(), 1),
                       **metricas(g.set_index("fecha")["t"], g.set_index("fecha")["t_era5"])}
                      for c, g in comp_t.groupby("codigo")])
if len(val_t):
    val_t["dif_alt_m"] = (val_t["altitud"] - val_t["alt_celda_era5"]).round(0)
    val_t["sesgo_esperado_6.5"] = (val_t["dif_alt_m"] * 6.5 / 1000).round(2)   # ERA5 más caliente si la estación está más alta
    guardar_tabla(val_t, "3_validacion_temperatura")
    display(val_t[["codigo", "nombre", "altitud", "alt_celda_era5", "dif_alt_m", "t_media_est",
                   "sesgo_abs", "sesgo_esperado_6.5", "r", "n_meses"]].round({"alt_celda_era5": 0}))
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ax = axes[0]
    for c, g in comp_t.groupby("codigo"):
        ax.scatter(g["t"], g["t_era5"], s=6, alpha=0.5, label=str(g["nombre"].iloc[0]).split("[")[0].strip().title())
    lo, hi = np.nanmin(comp_t[["t", "t_era5"]].values) - 1, np.nanmax(comp_t[["t", "t_era5"]].values) + 1
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set(xlim=(lo, hi), ylim=(lo, hi), xlabel="Temperatura media mensual · estación (°C)",
           ylabel="Temperatura media mensual · ERA5-Land (°C)", title="Temperatura: estación vs. ERA5-Land")
    ax.legend(fontsize=7, markerscale=3)
    ax = axes[1]
    ax.scatter(val_t["dif_alt_m"], val_t["sesgo_abs"], s=60, color="#d7301f", edgecolor="k", zorder=3)
    for r in val_t.itertuples():
        ax.annotate(str(r.nombre).split("[")[0].strip().title(), (r.dif_alt_m, r.sesgo_abs), fontsize=7, xytext=(3, 3), textcoords="offset points")
    xx = np.linspace(min(val_t["dif_alt_m"].min(), -100), max(val_t["dif_alt_m"].max(), 100), 10)
    ax.plot(xx, xx * 6.5 / 1000, "k--", lw=1, label="Diferencia esperada con −6,5 °C/km")
    ax.axhline(0, color="grey", lw=0.5); ax.axvline(0, color="grey", lw=0.5)
    ax.set(xlabel="Altitud de la estación − altitud media del cuadro de ERA5-Land (m)", ylabel="Diferencia ERA5-Land − estación (°C)",
           title="¿La diferencia de ERA5-Land se explica por la altura?")
    ax.legend()
    plt.tight_layout()
    guardar_fig(fig, "3_validacion_temperatura")
    plt.show()
else:
    print("Sin series de temperatura de estaciones: el cambio con la altura se estima solo con ERA5-Land.")

# Cambio con la altura según ERA5-Land dentro de la cuenca (promedio del periodo de referencia)
t_clim = (ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR").filterDate(f"{PERIODO_REF[0]}-01-01", f"{PERIODO_REF[1] + 1}-01-01")
          .select("temperature_2m").mean().subtract(273.15).rename("t"))
muestra = fc_a_df(ee.Image.cat(t_clim, dem).sample(geom_cuenca, scale=11132, geometries=False))
gradientes = {}
fig, ax = plt.subplots(figsize=(7, 5))
if len(muestra) > 3:
    b = np.polyfit(muestra["elev"], muestra["t"], 1)
    gradientes["ERA5-Land (celdas de la cuenca)"] = b[0] * 1000
    ax.scatter(muestra["elev"], muestra["t"], s=18, color="#fdae61", label=f"ERA5-Land: {b[0] * 1000:.1f} °C/km")
    xx = np.linspace(muestra["elev"].min(), muestra["elev"].max(), 10)
    ax.plot(xx, np.polyval(b, xx), color="#f46d43")
if len(val_t) >= 3:
    b = np.polyfit(val_t["altitud"], val_t["t_media_est"], 1)
    gradientes["Estaciones IDEAM"] = b[0] * 1000
    ax.scatter(val_t["altitud"], val_t["t_media_est"], s=50, color="#2b83ba", edgecolor="k",
               label=f"Estaciones: {b[0] * 1000:.1f} °C/km")
    xx = np.linspace(val_t["altitud"].min(), val_t["altitud"].max(), 10)
    ax.plot(xx, np.polyval(b, xx), color="#2b83ba")
ax.set(xlabel="Altitud (m)", ylabel="Temperatura media (°C)", title="Cambio de la temperatura con la altura (referencia: −6,5 °C/km)")
ax.legend()
guardar_fig(fig, "3_temperatura_altura")
plt.show()
for k, v in gradientes.items():
    print(f"Cambio de temperatura con la altura, {k}: {v:.2f} °C/km")

> **Para interpretar (grupos 3 y 5):** ¿la diferencia de CHIRPS cambia con la altura? ¿La temperatura de ERA5-Land se parece más a una estación de páramo o de valle? Si su área está a 3.500 m y el cuadro de ERA5-Land promedia 2.800 m, ¿cuánto deberían diferir (usando el cambio de temperatura por kilómetro calculado)?

---
## 4. Variabilidad climática (ENOS)

**Conceptos clave:** variaciones de un año a otro frente a cambio climático · anomalía (diferencia frente a lo normal) · índice ONI · efecto a distancia del océano Pacífico.

Se calcula cuánto llovió cada mes en la cuenca por encima o por debajo de lo normal (**anomalía**, en %; CHIRPS, promedio de la cuenca) frente al promedio 1991–2020, y se agrupa según la fase del ENOS (El Niño, Neutro, La Niña). Un mes cuenta como El Niño o La Niña cuando el índice pasa de ±0,5 °C durante al menos 5 trimestres seguidos (criterio de NOAA). Con `INDICE_ENOS = "RONI"` se usa el índice relativo, que descuenta el calentamiento general del trópico.

In [ ]:
# 4.1 Índice ENOS y lluvia mensual de la cuenca
indice = cargar_indice_enos(INDICE_ENOS)
fase = fases_enos(indice)

ppt_cuenca = leer_cache("chirps_mensual_cuenca")
if ppt_cuenca is None:
    print("Calculando la lluvia mensual promedio de la cuenca con CHIRPS…")
    s = serie_mensual_region(lambda d0, d1: ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterDate(d0, d1)
                             .select("precipitation").sum(), geom_cuenca, ANIO_INICIO_SAT, ANIO_FIN_EF, 5566, "ppt")
    ppt_cuenca = s.reset_index()
    guardar_tabla(ppt_cuenca, "chirps_mensual_cuenca")
enos = ppt_cuenca.set_index("fecha")[["ppt"]]
ref = enos.loc[f"{PERIODO_REF[0]}":f"{PERIODO_REF[1]}"]
clim = ref.groupby(ref.index.month)["ppt"].mean()
enos["clim"] = enos.index.month.map(clim)
enos["anom_pct"] = (enos["ppt"] / enos["clim"] - 1) * 100
enos["indice"] = indice.reindex(enos.index)
enos["fase"] = fase.reindex(enos.index)
enos = enos.dropna(subset=["fase"])
enos["temporada"] = enos.index.month.map({12: "DEF", 1: "DEF", 2: "DEF", 3: "MAM", 4: "MAM", 5: "MAM",
                                          6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"})
guardar_tabla(enos.reset_index(), "4_anomalias_enos")
print(enos["fase"].value_counts().rename("meses"))

In [ ]:
# 4.2 Promedios por fase ENOS
orden = ["El Niño", "Neutro", "La Niña"]
fig = plt.figure(figsize=(15, 9))
ax1 = fig.add_subplot(2, 2, 1)
ax1.bar(clim.index, clim.values, color="#6baed6")
ax1.set(xticks=range(1, 13), xticklabels=list("EFMAMJJASOND"), ylabel="mm/mes",
        title=f"Lluvia promedio por mes {PERIODO_REF[0]}–{PERIODO_REF[1]} (dos temporadas de lluvia al año)")

ax2 = fig.add_subplot(2, 1, 2)
ax2.bar(enos.index, enos["anom_pct"], width=25, color=enos["fase"].map(COLORES_FASE))
ax2b = ax2.twinx()
ax2b.plot(enos.index, enos["indice"], color="k", lw=0.8)
ax2b.set_ylabel(f"{INDICE_ENOS} (°C)")
ax2b.grid(False)
ax2.set(ylabel="Anomalía de lluvia (%)", title="Anomalía mensual de lluvia en la cuenca (color = fase ENOS)")
ax2.set_ylim(-150, 250)

ax3 = fig.add_subplot(2, 2, 2)
tab = (enos.groupby(["temporada", "fase"])["anom_pct"].median().unstack()
       .reindex(index=["DEF", "MAM", "JJA", "SON"], columns=orden))
tab.loc["Año"] = enos.groupby("fase")["anom_pct"].median().reindex(orden)
tab.plot.bar(ax=ax3, color=[COLORES_FASE[f] for f in orden], rot=0)
ax3.axhline(0, color="k", lw=0.8)
ax3.set(ylabel="Mediana de la anomalía (%)", xlabel="", title="Anomalía de lluvia por fase y temporada")
plt.tight_layout()
guardar_fig(fig, "4_promedios_enos")
plt.show()
display(tab.round(1))

# Correlación con retraso: ¿cuántos meses tarda la señal en llegar?
retraso = pd.DataFrame({"retraso_meses": range(0, 7),
                        "r": [enos["anom_pct"].corr(enos["indice"].shift(k)) for k in range(0, 7)]}).round(2)
display(retraso.set_index("retraso_meses").T)

In [ ]:
# 4.3 Señal ENOS en cada estación: lluvia medida (IDEAM) vs. CHIRPS en el mismo punto
def anomalia_por_fase(df, col):
    # Mediana de la anomalía mensual (%) por estación y fase, con el promedio del periodo de referencia
    d = df.dropna(subset=[col]).copy()
    ref = d[d["fecha"].dt.year.between(*PERIODO_REF)]
    clim = ref.groupby(["codigo", ref["fecha"].dt.month])[col].mean()
    d["clim"] = [clim.get((c, m), np.nan) for c, m in zip(d["codigo"], d["fecha"].dt.month)]
    d["anom"] = (d[col] / d["clim"] - 1) * 100
    d["fase"] = d["fecha"].map(fase)
    return d.groupby(["codigo", "fase"])["anom"].median().unstack().reindex(columns=["El Niño", "Neutro", "La Niña"])

fases_orden = ["El Niño", "Neutro", "La Niña"]
comun = chirps_m.merge(est_m, on=["codigo", "fecha"])   # mismos meses en ambas fuentes, para que la comparación sea justa
r_est = anomalia_por_fase(comun, "ppt") if len(comun) else pd.DataFrame(columns=fases_orden, dtype=float)
r_chi = anomalia_por_fase(comun, "ppt_chirps") if len(comun) else pd.DataFrame(columns=fases_orden, dtype=float)
# Respaldo: estaciones sin serie de DHIME -> señal de CHIRPS con todos sus meses
r_chi = r_chi.combine_first(anomalia_por_fase(chirps_m, "ppt_chirps"))
respaldo = sorted(set(puntos["codigo"]) - set(r_est.dropna(how="all").index))
if respaldo:
    print("Estaciones con respaldo CHIRPS (sin serie de estación):", ", ".join(respaldo))

tabla_enos = puntos[["codigo", "nombre", "altitud", "lon", "lat"]].set_index("codigo")
for f_ in fases_orden:
    tabla_enos[f"{f_} · estación"] = r_est[f_]
    tabla_enos[f"{f_} · CHIRPS"] = r_chi[f_]
    tabla_enos[f"{f_} · diferencia"] = r_chi[f_] - r_est[f_]
tabla_enos = tabla_enos.sort_values("altitud", ascending=False).reset_index()
guardar_tabla(tabla_enos, "4_enos_estacion_vs_chirps")
cols_tab = ["nombre", "altitud"] + [f"{f_} · {s_}" for f_ in fases_orden for s_ in ["estación", "CHIRPS", "diferencia"]]
print("Mediana de la anomalía mensual de lluvia (%) por fase ENOS, ordenada por altitud (diferencia = CHIRPS − estación):")
display(tabla_enos[cols_tab].round(1).set_index("nombre"))

# Mapas: fila 1 = estación, fila 2 = CHIRPS
fuentes = [("estación", "Estación IDEAM"), ("CHIRPS", "CHIRPS")]
fig, axes = plt.subplots(len(fuentes), 2, figsize=(13, 5.3 * len(fuentes)), squeeze=False)
norma = mcolors.TwoSlopeNorm(0, -40, 40)
for i, (suf, titulo) in enumerate(fuentes):
    for j, f_ in enumerate(["El Niño", "La Niña"]):
        ax = axes[i, j]
        mapa_base(ax, f"{titulo} · anomalía mediana en {f_} (%)")
        sc = ax.scatter(tabla_enos["lon"], tabla_enos["lat"], c=tabla_enos[f"{f_} · {suf}"], cmap="BrBG", norm=norma,
                        s=70, edgecolor="k", zorder=3)
        for r in tabla_enos.itertuples():
            ax.annotate(str(r.nombre).split("[")[0].strip().title(), (r.lon, r.lat), fontsize=6, xytext=(3, 3), textcoords="offset points")
        plt.colorbar(sc, ax=ax, shrink=0.7, label="%")
plt.tight_layout()
guardar_fig(fig, "4_mapa_enos_estacion_vs_chirps")
plt.show()

# Dispersión: cada punto es una estación en una fase
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
for f_ in fases_orden:
    ax.scatter(tabla_enos[f"{f_} · estación"], tabla_enos[f"{f_} · CHIRPS"], s=55, color=COLORES_FASE[f_],
               edgecolor="k", label=f_)
lim = np.nanmax(np.abs(tabla_enos[[c for c in tabla_enos if "·" in c and "diferencia" not in c]].values)) * 1.1
ax.plot([-lim, lim], [-lim, lim], "k--", lw=1)
ax.axhline(0, color="grey", lw=0.5); ax.axvline(0, color="grey", lw=0.5)
ax.set(xlim=(-lim, lim), ylim=(-lim, lim), xlabel="Anomalía según la estación (%)", ylabel="Anomalía según CHIRPS (%)",
       title="¿CHIRPS reproduce la señal ENOS medida en las estaciones?")
ax.legend()
ax = axes[1]
x = np.arange(len(tabla_enos))
for k, f_ in enumerate(["El Niño", "La Niña"]):
    ax.bar(x + (k - 0.5) * 0.38, tabla_enos[f"{f_} · diferencia"], width=0.38, color=COLORES_FASE[f_], label=f_)
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f"{str(n).split('[')[0].strip().title()}\n{a:.0f} m" for n, a in zip(tabla_enos["nombre"], tabla_enos["altitud"])],
                   rotation=60, ha="right", fontsize=7)
ax.set(ylabel="CHIRPS − estación (puntos porcentuales)", title="Diferencia en la anomalía por estación (de mayor a menor altitud)")
ax.legend()
plt.tight_layout()
guardar_fig(fig, "4_comparacion_enos_estacion_chirps")
plt.show()
for f_ in ["El Niño", "La Niña"]:
    e_, c_ = tabla_enos[f"{f_} · estación"], tabla_enos[f"{f_} · CHIRPS"]
    print(f"{f_}: mediana estaciones {e_.median():.1f} % · CHIRPS {c_.median():.1f} % · "
          f"diferencia absoluta media {(c_ - e_).abs().mean():.1f} puntos · correlación entre estaciones r = {e_.corr(c_):.2f}")

> **Para interpretar (grupos 2, 3 y 5):** ¿CHIRPS reproduce la intensidad de la señal ENOS que miden las estaciones, o la suaviza? ¿La diferencia cambia con la altitud? ¿Qué tan fuerte es la señal de La Niña frente a El Niño? ¿Es igual en todas las temporadas? Si su pregunta es sobre **cambios de un año a otro**, los índices ENOS (ONI, RONI) y los promedios por fase son la herramienta adecuada; los índices de extremos de la siguiente sección responden mejor a preguntas sobre **extremos y cambio a largo plazo**.

---
## 5. Extremos y tendencias

**Conceptos clave:** eventos extremos · índices de extremos (ETCCDI) · tendencia y qué tan confiable es.

| Índice | Qué mide | Unidad |
|---|---|---|
| PRCPTOT | Lluvia anual en días húmedos (≥ 1 mm) | mm |
| Rx1day / Rx5day | Máxima lluvia en 1 día / en 5 días seguidos | mm |
| R95p | Lluvia anual en los días más lluviosos (los que superan al 95 % de los días de lluvia de 1991–2020) | mm |
| R20mm | Días con lluvia ≥ 20 mm | días |
| CDD | Máxima racha de días secos (< 1 mm) | días |
| TXx | Máxima temperatura diaria del año (ERA5-Land) | °C |

En cada punto se usa la **serie de la estación** si tiene al menos `UMBRAL_DATOS` % de días con dato y 20 años útiles; si no, se usa **CHIRPS**. La tendencia se evalúa con la prueba de **Mann-Kendall** (se considera clara si p < 0,05) y su tamaño con la **pendiente de Sen**, expresada como cambio por década.

In [ ]:
# 5.1 Índices ETCCDI anuales por punto
PERIODO_TENDENCIA = (PERIODO_REF[0], ANIO_FIN_EF)   # 🔧
UNIDADES = {"PRCPTOT": "mm", "Rx1day": "mm", "Rx5day": "mm", "R95p": "mm", "R20mm": "días", "CDD": "días", "TXx": "°C"}
dias = pd.date_range(f"{PERIODO_TENDENCIA[0]}-01-01", f"{PERIODO_TENDENCIA[1]}-12-31")
ppt_est = series[series["variable"] == "ppt"]
partes = []
for p in puntos.itertuples():
    f_inv = inv[inv["codigo"] == p.codigo] if len(inv) else pd.DataFrame()
    usa_est = len(f_inv) > 0 and f_inv["pct_ppt"].iloc[0] >= UMBRAL_DATOS and f_inv["anios_utiles_ppt"].iloc[0] >= 20
    if usa_est:
        s, fuente = ppt_est[ppt_est["codigo"] == p.codigo].set_index("fecha")["valor"], "Estación IDEAM"
    else:
        s, fuente = chirps_d[chirps_d["codigo"] == p.codigo].set_index("fecha")["ppt"], "CHIRPS"
    s = s[~s.index.duplicated()].reindex(dias)
    base = s[f"{PERIODO_REF[0]}":f"{PERIODO_REF[1]}"]
    idx = indices_etccdi(s, p95=base[base >= 1].quantile(0.95))
    if len(idx):
        partes.append(idx.reset_index().assign(codigo=p.codigo, fuente=fuente))
etccdi = pd.concat(partes, ignore_index=True)

if CALCULAR_TXX:
    tx = leer_cache("era5_tmax_diaria_puntos")
    if tx is None or set(puntos["codigo"]) - set(tx["codigo"]):
        tx = serie_en_puntos(ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR"), ["temperature_2m_max"], puntos, 11132,
                             f"{PERIODO_TENDENCIA[0]}-01-01", f"{PERIODO_TENDENCIA[1]}-12-31", dias_trozo=730,
                             etiqueta="ERA5-Land Tmáx")
        guardar_tabla(tx, "era5_tmax_diaria_puntos")
    txx = (tx.assign(anio=tx["fecha"].dt.year).groupby(["codigo", "anio"])["temperature_2m_max"].max() - 273.15).rename("TXx")
    etccdi = etccdi.merge(txx.reset_index(), on=["codigo", "anio"], how="left")
guardar_tabla(etccdi, "5_indices_etccdi")
print(etccdi["fuente"].value_counts().rename("puntos-año por fuente"))
display(etccdi.groupby("anio")[[c for c in UNIDADES if c in etccdi]].median().round(1).tail(10))

In [ ]:
# 5.2 Tendencias (Mann-Kendall y pendiente de Sen) y mapa
indices = [c for c in UNIDADES if c in etccdi]
filas = []
for c, g in etccdi.groupby("codigo"):
    g = g.set_index("anio")
    for i in indices:
        filas.append({"codigo": c, "indice": i, "fuente": "ERA5-Land" if i == "TXx" else g["fuente"].iloc[0], **tendencia(g[i])})
tend = pd.DataFrame(filas).merge(puntos[["codigo", "nombre", "lon", "lat", "altitud"]], on="codigo")
guardar_tabla(tend, "5_tendencias")

sig = tend["p"] < 0.05
resumen = pd.DataFrame({
    "puntos": tend.groupby("indice").size(),
    "aumento claro": tend[sig & (tend["tendencia"] == "aumento")].groupby("indice").size(),
    "disminución clara": tend[sig & (tend["tendencia"] == "disminución")].groupby("indice").size(),
    "mediana Sen por década": tend.groupby("indice")["sen_por_decada"].median().round(2),
}).fillna(0).reindex(indices)
resumen["unidad"] = [f"{UNIDADES[i]}/década" for i in resumen.index]
display(resumen)

i = INDICE_TENDENCIA if INDICE_TENDENCIA in indices else "Rx1day"
d = tend[tend["indice"] == i]
vmax = max(np.nanmax(np.abs(d["sen_por_decada"])), 1e-6)
cmap = {"CDD": "BrBG_r", "TXx": "RdBu_r"}.get(i, "BrBG")
norm = mcolors.TwoSlopeNorm(0, -vmax, vmax)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1.1, 1]})
mapa_base(axes[0], f"Pendiente de Sen de {i} ({UNIDADES[i]}/década), {PERIODO_TENDENCIA[0]}–{PERIODO_TENDENCIA[1]}")
s_ = d["p"] < 0.05
sc = axes[0].scatter(d.loc[s_, "lon"], d.loc[s_, "lat"], c=d.loc[s_, "sen_por_decada"], cmap=cmap, norm=norm,
                     s=90, edgecolor="k", linewidth=1.3, zorder=4, label="tendencia clara (p < 0,05)")
axes[0].scatter(d.loc[~s_, "lon"], d.loc[~s_, "lat"], c=d.loc[~s_, "sen_por_decada"], cmap=cmap, norm=norm,
                s=35, edgecolor="#9e9e9e", zorder=3, label="sin tendencia clara")
plt.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes[0], shrink=0.7, label=f"{UNIDADES[i]}/década")
axes[0].legend(loc="lower left", fontsize=8)

med = etccdi.groupby("anio")[i].median().dropna()
t = tendencia(med)
axes[1].plot(med.index, med.values, "o-", ms=3, color="#2b8cbe")
if not np.isnan(t["sen_por_decada"]):
    r = mk.original_test(med.values)
    axes[1].plot(med.index, r.intercept + r.slope * np.arange(len(med)), "r--",
                 label=f"Sen = {t['sen_por_decada']} {UNIDADES[i]}/década · p = {t['p']}")
    axes[1].legend()
axes[1].set(xlabel="Año", ylabel=f"{i} ({UNIDADES[i]})", title=f"{i}: mediana de los puntos de la cuenca")
plt.tight_layout()
guardar_fig(fig, f"5_tendencia_{i}")
plt.show()

> **Para interpretar (grupos 3, 5 y 6):** ¿cuántos puntos muestran tendencias claras (p < 0,05)? ¿Cambia la conclusión si se usa la serie de la estación en lugar de CHIRPS, o si se cambia el periodo? Una tendencia de 30–40 años puede ser parte de ciclos naturales de varios años: ¿qué índices responden realmente a su pregunta y cuáles sobran?

---
## 6. Embalses y lagunas

**Conceptos clave:** medir el agua con imágenes de satélite (NDWI) · series históricas · cuerpos de agua como señal de alerta.

- **JRC Global Surface Water** clasifica cada mes de 1984 a 2021 cada cuadro de 30 m de las imágenes Landsat como agua, no agua o sin observación (nubes). Solo se usan los meses en los que se observó al menos el 80 % del área del cuerpo de agua.
- **Sentinel-2** (10–20 m, desde 2017) extiende la serie hasta hoy con imágenes combinadas cada tres meses del índice de agua **NDWI** (agua si NDWI > 0), descartando nubes y sombras.

Las franjas de color de fondo marcan los periodos de El Niño (rojo) y La Niña (azul). Ningún paso requiere permisos de campo.

In [ ]:
# 6.1 Área inundada mensual (JRC) y trimestral (Sentinel-2)
def area_mensual_jrc(geom):
    pix = ee.Image.pixelArea().divide(1e6)
    def f(img):
        w = img.select("water")
        st = ee.Image.cat(pix.updateMask(w.eq(2)).rename("agua"), pix.updateMask(w.gt(0)).rename("observado"),
                          pix.rename("total")).reduceRegion(ee.Reducer.sum(), geom, 30, maxPixels=1e10)
        return ee.Feature(None, st).set("fecha", img.date().format("YYYY-MM-dd"))
    return fc_a_df(ee.FeatureCollection(ee.ImageCollection("JRC/GSW1_4/MonthlyHistory").map(f)))

def area_s2_trimestral(geom, inicio="2017-04-01"):
    pix = ee.Image.pixelArea().divide(1e6)
    col0 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(geom)
    def limpiar(img):   # conserva vegetación, suelo, agua y no clasificado (SCL 4, 5, 6, 7)
        return img.updateMask(img.select("SCL").remap([4, 5, 6, 7], [1, 1, 1, 1], 0))
    n = len(pd.date_range(inicio, f"{ANIO_FIN_EF}-12-31", freq="QS-JAN"))
    def f(k):
        d0 = ee.Date(inicio).advance(ee.Number(k).multiply(3), "month")
        col = col0.filterDate(d0, d0.advance(3, "month")).map(limpiar)
        ndwi = col.median().normalizedDifference(["B3", "B8"])
        st = ee.Image.cat(pix.updateMask(ndwi.gt(0)).rename("agua"), pix.updateMask(ndwi.mask()).rename("observado"),
                          pix.rename("total")).reduceRegion(ee.Reducer.sum(), geom, 20, maxPixels=1e10)
        d = ee.Dictionary(ee.Algorithms.If(col.size().gt(0), st, ee.Dictionary({})))
        return ee.Feature(None, d).set({"fecha": d0.format("YYYY-MM-dd"), "n_imagenes": col.size()})
    return fc_a_df(ee.FeatureCollection(ee.List.sequence(0, n - 1).map(f)))

def preparar(df, fuente):
    df = df.copy()
    for c in ["agua", "observado", "total"]:
        df[c] = pd.to_numeric(df.get(c), errors="coerce")
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["frac_observada"] = df["observado"] / df["total"]
    df["area_km2"] = df["agua"].where(df["frac_observada"] >= 0.8)
    return df.assign(fuente=fuente)

agua = leer_cache("6_area_cuerpos_agua")
if agua is None and not geom_emb:
    print("⚠️ No se encontraron cuerpos de agua; revise EMBALSES en la celda 0.2.")
    agua = pd.DataFrame(columns=["nombre", "fecha", "fuente", "area_km2", "frac_observada"])
elif agua is None:
    partes = []
    for nombre, g in geom_emb.items():
        print(f"  {nombre}: JRC mensual y Sentinel-2 trimestral…")
        zona = g.buffer(150)
        partes.append(preparar(area_mensual_jrc(zona), "JRC (Landsat)").assign(nombre=nombre))
        partes.append(preparar(area_s2_trimestral(zona), "Sentinel-2").assign(nombre=nombre))
    agua = pd.concat(partes, ignore_index=True)
    guardar_tabla(agua, "6_area_cuerpos_agua")
agua["fecha"] = pd.to_datetime(agua["fecha"])

res_agua = []
for nombre, g in agua.groupby("nombre"):
    j = g[g["fuente"].str.startswith("JRC")]
    ref = j[j["fecha"].dt.year.between(*PERIODO_REF)]["area_km2"]
    s2 = g[g["fuente"] == "Sentinel-2"].dropna(subset=["area_km2"])
    res_agua.append({"cuerpo de agua": nombre,
                     "meses JRC con ≥ 80 % observado (%)": round(j["area_km2"].notna().mean() * 100, 0),
                     "área mediana 1991–2020 (km²)": round(ref.median(), 2),
                     "área mínima JRC (km²)": round(j["area_km2"].min(), 2),
                     "fecha del mínimo": j.loc[j["area_km2"].idxmin(), "fecha"].strftime("%Y-%m") if j["area_km2"].notna().any() else "",
                     "último dato Sentinel-2 (km²)": round(s2["area_km2"].iloc[-1], 2) if len(s2) else np.nan,
                     "fecha último dato": s2["fecha"].iloc[-1].strftime("%Y-%m") if len(s2) else ""})
res_agua = pd.DataFrame(res_agua)
guardar_tabla(res_agua, "6_resumen_cuerpos_agua")
display(res_agua)

In [ ]:
# 6.2 Series de área con fases ENOS
def sombrear_enos(ax, inicio, fin):
    f = fase[inicio:fin]
    for _, g in f.groupby((f != f.shift()).cumsum()):
        if g.iloc[0] != "Neutro":
            ax.axvspan(g.index[0], g.index[-1] + pd.offsets.MonthEnd(1), color=COLORES_FASE[g.iloc[0]], alpha=0.15, lw=0)

nombres = list(agua["nombre"].unique())
if not nombres:
    print("Sin series de cuerpos de agua para graficar.")
else:
    fig, axes = plt.subplots(len(nombres), 1, figsize=(14, 3.2 * len(nombres)), sharex=True, squeeze=False)
    for ax, nombre in zip(axes[:, 0], nombres):
        g = agua[agua["nombre"] == nombre]
        sombrear_enos(ax, "1984-01-01", f"{ANIO_FIN_EF}-12-31")
        j = g[g["fuente"].str.startswith("JRC")].set_index("fecha")["area_km2"]
        ax.plot(j.index, j.values, ".", ms=3, color="#6baed6", label="JRC mensual")
        ax.plot(j.index, j.rolling(12, min_periods=4).median(), color="#08519c", lw=1.5, label="JRC mediana móvil 12 meses")
        s2 = g[g["fuente"] == "Sentinel-2"].set_index("fecha")["area_km2"]
        ax.plot(s2.index, s2.values, "s-", ms=4, color="#e6550d", lw=1, label="Sentinel-2 trimestral")
        ax.set(ylabel="km²", title=nombre)
    axes[0, 0].legend(loc="upper left", fontsize=8, ncol=3)
    plt.tight_layout()
    guardar_fig(fig, "6_series_cuerpos_agua")
    plt.show()

> **Para interpretar (grupo 2):** ¿qué porcentaje de meses se pierde por nubes? ¿Los mínimos de área coinciden con El Niño (por ejemplo 1991–1992, 1997–1998, 2009–2010, 2015–2016)? En un embalse, ¿cómo separar el efecto del clima del manejo del embalse? Para usar sus propias lagunas, agregue sus coordenadas en `EMBALSES` (celda 0.2).

---
## 7. Índice de riesgo mínimo

**Conceptos clave:** marco de riesgo del IPCC AR6 · poner los indicadores en la misma escala (0 a 1) · índice que combina varios indicadores.

Un indicador por componente, calculado para cada subcuenca:

| Componente | Opciones (parámetro en 0.2) | Fuente | Cómo se lee |
|---|---|---|---|
| Peligro | `dias_secos`: días al año con lluvia < 1 mm · `rx1day`: máximo anual de lluvia diaria · `ppt_anual`: lluvia anual | CHIRPS 1991–2020 | más días secos o más intensidad = más peligro; menos lluvia = más peligro |
| Exposición | `poblacion`: habitantes por km² · `cultivos_urbano`: proporción de cultivos y área construida | WorldPop 2020 · ESA WorldCover | más = más expuesto |
| Vulnerabilidad | `sin_cobertura_natural`: proporción del área sin bosque, arbustal, pastizal, humedal o agua · `archivo`: un indicador propio por subcuenca | ESA WorldCover · CSV propio (por ejemplo, IPM o NBI del DANE) | más = más vulnerable |

Cada indicador se pasa a una escala de 0 a 1 (0 = el valor más bajo de la cuenca, 1 = el más alto) y se combinan con un promedio con pesos. Es un **índice relativo**: ordena las subcuencas entre sí, no mide un riesgo absoluto. Para `VULNERABILIDAD = "archivo"` suba a `CARPETA_DATOS` el archivo `vulnerabilidad_subcuencas.csv` con las columnas `subcuenca` y `valor`.

In [ ]:
# 7.1 Indicadores por subcuenca (Earth Engine)
ind = leer_cache("7_indicadores_subcuencas", fechas=())
if ind is None:
    print("Calculando indicadores por subcuenca…")
    ini, fin = PERIODO_REF
    n_anios = fin - ini + 1
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterDate(f"{ini}-01-01", f"{fin + 1}-01-01").select("precipitation")
    dias_secos = chirps.map(lambda i: i.lt(1)).sum().divide(n_anios).rename("dias_secos")
    rx1day = ee.ImageCollection(ee.List.sequence(ini, fin).map(
        lambda y: chirps.filter(ee.Filter.calendarRange(y, y, "year")).max())).mean().rename("rx1day")
    ppt_anual = chirps.sum().divide(n_anios).rename("ppt_anual")
    wc = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
    natural = wc.remap([10, 20, 30, 80, 90, 95, 100], [1, 1, 1, 1, 1, 1, 1], 0)
    wdpa = ee.FeatureCollection("WCMC/WDPA/current/polygons").filterBounds(geom_cuenca)
    fracciones = ee.Image.cat(wc.eq(40).Or(wc.eq(50)).rename("cultivos_urbano"),
                              natural.eq(0).rename("sin_cobertura_natural"),
                              ee.Image(0).byte().paint(wdpa, 1).rename("frac_protegida"))
    pob = (ee.ImageCollection("WorldPop/GP/100m/pop").filter(ee.Filter.eq("year", 2020))
           .filterBounds(geom_cuenca).mosaic().rename("poblacion"))
    base = fc_sub.select(["HYBAS_ID"])
    d1 = fc_a_df(ee.Image.cat(dias_secos, rx1day, ppt_anual).reduceRegions(base, ee.Reducer.mean(), 5566)
                 .select(["HYBAS_ID", "dias_secos", "rx1day", "ppt_anual"], None, False))
    d2 = fc_a_df(fracciones.reduceRegions(base, ee.Reducer.mean(), 100)
                 .select(["HYBAS_ID", "cultivos_urbano", "sin_cobertura_natural", "frac_protegida"], None, False))
    d3 = fc_a_df(pob.reduceRegions(base, ee.Reducer.sum(), 100).select(["HYBAS_ID", "sum"], None, False)).rename(columns={"sum": "poblacion"})
    ind = d1.merge(d2, on="HYBAS_ID").merge(d3, on="HYBAS_ID")
    ind["HYBAS_ID"] = ind["HYBAS_ID"].astype("int64")
    ind = gdf_sub[["HYBAS_ID", "subcuenca", "area_km2", "elev_media"]].merge(ind, on="HYBAS_ID")
    ind["densidad_hab_km2"] = ind["poblacion"] / ind["area_km2"]
    guardar_tabla(ind, "7_indicadores_subcuencas")
ind["HYBAS_ID"] = ind["HYBAS_ID"].astype("int64")
if VULNERABILIDAD == "archivo":
    v = leer_tabla(os.path.join(CARPETA_DATOS, "vulnerabilidad_subcuencas.csv"))
    ind = ind.drop(columns="archivo", errors="ignore").merge(v.rename(columns={"valor": "archivo"})[["subcuenca", "archivo"]],
                                                              on="subcuenca", how="left")
display(ind.round(2))

In [ ]:
# 7.2 Escala de 0 a 1, índice de riesgo y mapas
DIRECCION = {"dias_secos": 1, "rx1day": 1, "ppt_anual": -1, "poblacion": 1, "cultivos_urbano": 1,
             "sin_cobertura_natural": 1, "archivo": 1}

def valor(tabla, nombre):
    if nombre == "poblacion":
        v = tabla["densidad_hab_km2"]
        return np.log10(1 + v) if EXPOSICION_LOG else v
    return tabla[nombre]

def minmax(v, direccion=1):
    v = v.astype(float)
    n = (v - v.min()) / (v.max() - v.min()) if v.max() > v.min() else v * 0 + 0.5
    return n if direccion > 0 else 1 - n

def indice_riesgo(tabla, peligro, exposicion, vulnerabilidad, pesos=PESOS, metodo=METODO_INDICE):
    comp = pd.DataFrame({"peligro": minmax(valor(tabla, peligro), DIRECCION[peligro]),
                         "exposicion": minmax(valor(tabla, exposicion), DIRECCION[exposicion]),
                         "vulnerabilidad": minmax(valor(tabla, vulnerabilidad), DIRECCION[vulnerabilidad])})
    w = pd.Series(pesos)[comp.columns].astype(float)
    if metodo == "geometrico":
        r = np.exp((np.log(comp.clip(lower=0.01)) * w).sum(axis=1) / w.sum())
    else:
        r = (comp * w).sum(axis=1) / w.sum()
    return comp, r

comp_n, ind["indice_riesgo"] = indice_riesgo(ind, PELIGRO, EXPOSICION, VULNERABILIDAD)
ind[["P_norm", "E_norm", "V_norm"]] = comp_n.values
ind["clase"] = pd.qcut(ind["indice_riesgo"].rank(method="first"), 3, labels=["Bajo", "Medio", "Alto"]) if len(ind) >= 3 else "—"
ind["rango"] = ind["indice_riesgo"].rank(ascending=False).astype(int)
riesgo = gdf_sub[["HYBAS_ID", "geometry"]].merge(ind, on="HYBAS_ID")
guardar_tabla(ind.sort_values("rango"), "7_indice_riesgo")
display(ind.sort_values("rango")[["rango", "subcuenca", "area_km2", PELIGRO,
                                  "densidad_hab_km2" if EXPOSICION == "poblacion" else EXPOSICION, VULNERABILIDAD,
                                  "P_norm", "E_norm", "V_norm", "indice_riesgo", "clase"]].round(2))

fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
for ax, col, titulo, cmap in [(axes[0], "P_norm", f"Peligro: {PELIGRO}", "YlOrBr"),
                              (axes[1], "E_norm", f"Exposición: {EXPOSICION}", "PuRd"),
                              (axes[2], "V_norm", f"Vulnerabilidad: {VULNERABILIDAD}", "BuPu"),
                              (axes[3], "indice_riesgo", "Índice de riesgo relativo", "RdYlGn_r")]:
    riesgo.plot(column=col, cmap=cmap, vmin=0, vmax=1, ax=ax, edgecolor="#616161", lw=0.4, legend=True,
                legend_kwds={"shrink": 0.6})
    cuenca_gdf.boundary.plot(ax=ax, color="black", lw=1)
    ax.set_title(titulo, fontsize=10)
    ax.set_axis_off()
for r in riesgo.itertuples():
    pt = r.geometry.representative_point()
    axes[3].annotate(r.subcuenca, (pt.x, pt.y), fontsize=6, ha="center")
plt.suptitle(f"Riesgo hidroclimático relativo por subcuenca · {CUENCA_NOMBRE} (IPCC AR6, un indicador por componente)")
plt.tight_layout()
guardar_fig(fig, "7_mapa_riesgo")
plt.show()

m = mapa_folium(centro.y, centro.x)
folium.Choropleth(riesgo[["subcuenca", "indice_riesgo", "geometry"]], data=riesgo, columns=["subcuenca", "indice_riesgo"],
                  key_on="feature.properties.subcuenca", fill_color="RdYlGn_r", fill_opacity=0.7, line_weight=0.5,
                  legend_name="Índice de riesgo relativo").add_to(m)
folium.GeoJson(riesgo[["subcuenca", "indice_riesgo", "clase", "geometry"]].assign(indice_riesgo=lambda d: d["indice_riesgo"].round(2),
                                                                                   clase=lambda d: d["clase"].astype(str)),
               style_function=lambda f: {"fillOpacity": 0, "weight": 0},
               tooltip=folium.GeoJsonTooltip(["subcuenca", "indice_riesgo", "clase"])).add_to(m)
folium.LayerControl().add_to(m)
m.save(os.path.join(CARPETA_SALIDA, "7_mapa_riesgo.html"))
m

In [ ]:
# 7.3 ¿Cambian las subcuencas prioritarias si cambio de indicador?
filas = []
for pel in ["dias_secos", "rx1day", "ppt_anual"]:
    for exp in ["poblacion", "cultivos_urbano"]:
        _, r = indice_riesgo(ind, pel, exp, VULNERABILIDAD)
        filas.append({"peligro": pel, "exposicion": exp,
                      "3 subcuencas de mayor riesgo": ", ".join(ind.assign(r=r).nlargest(3, "r")["subcuenca"]),
                      "correlación de rangos con la selección actual": round(r.rank().corr(ind["indice_riesgo"].rank()), 2)})
sens = pd.DataFrame(filas)
guardar_tabla(sens, "7_comparacion_indicadores")
display(sens)

In [ ]:
# 7.4 Cruce con áreas protegidas y municipios (grupo 4: ¿quién responde por las subcuencas prioritarias?)
top = ind.nsmallest(3, "rango")["subcuenca"].tolist()
mun = fc_a_gdf(ee.FeatureCollection("FAO/GAUL/2015/level2").filterBounds(geom_cuenca)
               .map(lambda f: f.simplify(200)).select(["ADM1_NAME", "ADM2_NAME"]))
mun["geometry"] = mun.buffer(0)
cruce = gpd.overlay(gdf_sub[gdf_sub["subcuenca"].isin(top)][["subcuenca", "area_km2", "geometry"]],
                    mun[["ADM1_NAME", "ADM2_NAME", "geometry"]], how="intersection")
cruce["km2"] = cruce.to_crs(CRS_PLANO).area / 1e6
cruce["% de la subcuenca"] = (cruce["km2"] / cruce["area_km2"] * 100).round(1)
cruce = cruce[cruce["% de la subcuenca"] >= 1].rename(columns={"ADM1_NAME": "departamento", "ADM2_NAME": "municipio"})

wd = fc_a_gdf(ee.FeatureCollection("WCMC/WDPA/current/polygons").filterBounds(geom_cuenca)
              .map(lambda f: f.simplify(100)).select(["NAME", "DESIG", "GOV_TYPE"]))
if len(wd):
    wd["geometry"] = wd.buffer(0)
    prot = gpd.overlay(gdf_sub[gdf_sub["subcuenca"].isin(top)][["subcuenca", "area_km2", "geometry"]],
                       wd[["NAME", "DESIG", "geometry"]], how="intersection")
    prot["% de la subcuenca"] = (prot.to_crs(CRS_PLANO).area / 1e6 / prot["area_km2"] * 100).round(1)
    prot = prot[prot["% de la subcuenca"] >= 0.5].rename(columns={"NAME": "área protegida", "DESIG": "designación"})
else:
    prot = pd.DataFrame(columns=["subcuenca", "área protegida", "designación", "% de la subcuenca"])

print("Subcuencas de mayor riesgo:", ", ".join(top))
print("\nMunicipios con jurisdicción (≥ 1 % del área de la subcuenca):")
display(cruce[["subcuenca", "departamento", "municipio", "% de la subcuenca"]].sort_values(["subcuenca", "% de la subcuenca"], ascending=[True, False]))
print("\nÁreas protegidas (WDPA, incluye áreas del RUNAP):")
display(prot[["subcuenca", "área protegida", "designación", "% de la subcuenca"]].sort_values(["subcuenca", "% de la subcuenca"], ascending=[True, False]))
print("\nFracción protegida por subcuenca (%):")
display((ind.set_index("subcuenca")["frac_protegida"] * 100).round(1).rename("% protegido").to_frame().T)
guardar_tabla(cruce.drop(columns="geometry"), "7_municipios_subcuencas_prioritarias")
guardar_tabla(prot.drop(columns="geometry", errors="ignore"), "7_areas_protegidas_subcuencas_prioritarias")

> **Para interpretar (grupos 1, 4 y 6):** ¿qué subcuencas cambian de posición al cambiar un indicador o los pesos? ¿Qué tan bien representa cada indicador a su componente? ¿Qué información de vulnerabilidad y de las instituciones se puede sacar de datos ya publicados antes de ir a campo, y cuál requiere entrevistas?

---
## 8. Su propuesta: qué necesita del notebook

**Conceptos clave:** qué se puede hacer con lo disponible · datos que hay frente a datos que se necesitan · cambiar una fuente por otra.

Las secciones 1 a 7 son un caso completo para la cuenca del río Bogotá. Su propuesta no necesita todo: el grupo decide **qué partes le sirven, qué debe cambiar y qué datos propios agrega**, y vuelve a ejecutar solo eso. No hay una respuesta única.

1. Lea el comentario de la evaluación y la pregunta guía de su grupo (8.1).
2. Revise qué resultados produjo el notebook (8.2) y vuelva a las secciones que respondan a su pregunta.
3. Cambie parámetros (celda 0.2) o agregue datos, ejecute de nuevo (**Entorno de ejecución → Ejecutar celda seleccionada y siguientes**) y registre lo que usó y lo que cambió (8.3). Si cambia la cuenca o el periodo, ponga `REUTILIZAR_DESCARGAS = False`.

**Datos propios que el notebook sabe leer** (súbalos a la sesión y ejecute de nuevo desde la celda 0.3):

| Dato | Formato |
|---|---|
| Series de otras estaciones IDEAM de la cuenca | CSV de DHIME tal como se descarga (`PTPM_CON`, `TSSM_MEDIA_D`); las estaciones con serie entran como puntos de análisis |
| Lagunas o embalses | `nombre: (lon, lat, radio_m)` en `EMBALSES` (celda 0.2) |
| Vulnerabilidad por subcuenca | `vulnerabilidad_subcuencas.csv` con columnas `subcuenca` y `valor`, y `VULNERABILIDAD = "archivo"` |

Cualquier otro dato (una capa, una tabla de su proyecto) lo puede leer con `pd.read_csv` o `gpd.read_file` y cruzarlo con las tablas del notebook.

In [ ]:
# 8.1 Comentario de la evaluación y pregunta guía (cambie GRUPO en la celda 0.1)
from IPython.display import Markdown
GUIA = {
    1: ("Chingaza", "Propuesta muy amplia; entrevistas; evaluación de la gestión de las instituciones",
        "¿Qué parte de la vulnerabilidad y de la gestión de las instituciones se puede estimar con datos ya publicados antes de ir a campo?"),
    2: ("Lagos centinela", "Muchas variables; trámites ante autoridades",
        "¿Qué variables se pueden medir con imágenes, sin permisos de campo, y cuáles quedan por fuera?"),
    3: ("Atacazo", "¿La temperatura usada representa el área?; índices de extremos (ETCCDI) frente a índices ENOS",
        "¿La fuente de temperatura representa bien su área? ¿Su pregunta es de variabilidad, de cambio o de ambas?"),
    4: ("Gobernanza fluvial", "Sin sugerencias",
        "¿Qué actores (municipios, autoridades ambientales, áreas protegidas) responden por las zonas de mayor riesgo?"),
    5: ("Manzanares", "Alcance; datos disponibles en fuentes oficiales",
        "¿Las fuentes oficiales tienen la información que su propuesta requiere? ¿Qué variables se sustituyen por satélite?"),
    6: ("Cuencas andinas", "Alcance de las cuatro etapas",
        "¿Cómo se ven sus cuatro etapas reducidas a un indicador cada una? ¿Cuál se puede cumplir en el tiempo disponible?"),
}
grupos = [GRUPO] if GRUPO in GUIA else list(GUIA)
display(Markdown("\n".join(f"### Grupo {g} · {GUIA[g][0]}\n**Comentario de la evaluación:** {GUIA[g][1]}\n\n"
                           f"**Pregunta guía:** {GUIA[g][2]}\n" for g in grupos)))

In [ ]:
# 8.2 Resultados disponibles: tablas principales en memoria y archivos guardados
TABLAS = [  # (variable, sección, contenido)
    ("tabla_sub", "1.1", "Subcuencas: área y relieve"),
    ("gdf_sub", "1.1", "Subcuencas con geometría (GeoDataFrame)"),
    ("inv", "2.3", "Estaciones de la cuenca: datos esperados, datos reales y selección"),
    ("series", "2.2", "Series diarias de las estaciones (codigo, fecha, variable, valor)"),
    ("puntos", "3.1", "Puntos de análisis con altitud del catálogo, del DEM y del cuadro de ERA5-Land"),
    ("chirps_d", "3.1", "CHIRPS diario en cada punto"),
    ("era_m", "3.1", "ERA5-Land mensual en cada punto (temperatura y lluvia)"),
    ("comp", "3.2", "Pares mensuales de lluvia observada y CHIRPS"),
    ("val_ppt", "3.2", "Métricas de precipitación por estación (r, sesgo, RMSE, KGE)"),
    ("val_t", "3.3", "Métricas de temperatura por estación frente a ERA5-Land"),
    ("enos", "4.1", "Lluvia mensual de la cuenca, anomalía, índice y fase ENOS"),
    ("tabla_enos", "4.3", "Anomalía por fase en cada estación: estación vs. CHIRPS"),
    ("etccdi", "5.1", "Índices ETCCDI anuales por punto"),
    ("tend", "5.2", "Tendencias (Mann-Kendall, Sen) por punto e índice"),
    ("agua", "6.1", "Área mensual de cada cuerpo de agua (JRC y Sentinel-2)"),
    ("res_agua", "6.1", "Resumen por cuerpo de agua"),
    ("ind", "7.2", "Indicadores, componentes en escala 0–1 e índice de riesgo por subcuenca"),
    ("sens", "7.3", "Cómo cambia el orden de las subcuencas con otros indicadores"),
    ("cruce", "7.4", "Municipios en las subcuencas prioritarias"),
]
_g = globals()
display(pd.DataFrame([{"variable": n, "sección": s_, "contenido": t, "filas": len(_g[n]),
                       "columnas": ", ".join(map(str, _g[n].columns[:10])) + (" …" if _g[n].shape[1] > 10 else "")}
                      for n, s_, t in TABLAS if isinstance(_g.get(n), pd.DataFrame)]))
if "gradientes" in _g:
    print("gradientes (3.3):", {k: round(float(v), 2) for k, v in gradientes.items()}, "°C/km")
_arch = sorted(os.path.basename(f) for f in glob.glob(os.path.join(CARPETA_SALIDA, "*")))
print("\nFiguras y mapas guardados:", ", ".join(f for f in _arch if f.endswith((".png", ".html"))))
print("\nTablas guardadas (CSV):", ", ".join(f for f in _arch if f.endswith(".csv")))

In [ ]:
# 8.3 Lo que su grupo usa del notebook 🔧
# Una fila por cada pregunta o componente de SU propuesta.
SELECCION = [
    # (pregunta o componente de su propuesta, sección o tabla que usa, qué cambió (parámetros o datos), por qué le sirve)
]

sufijo = f"grupo{GRUPO}" if GRUPO else "ejemplo"
if not SELECCION:
    print("⚠️ Complete SELECCION con lo que su grupo usa del notebook y vuelva a ejecutar esta celda.")
else:
    seleccion = pd.DataFrame(SELECCION, columns=["pregunta_o_componente", "seccion_o_tabla", "cambios", "por_que"]).assign(grupo=GRUPO)
    guardar_tabla(seleccion, f"8_seleccion_{sufijo}")
    display(seleccion)

---
## 9. Hallazgos: lo que encontró su grupo

**Conceptos clave:** cifras para decidir qué hacer · mantener, sustituir o recortar.

Esta sección la construye el grupo con los resultados que eligió en la sección 8. Lo que presenta es: la **tabla de hallazgos** (9.1), una **figura propia** (9.2), la **lámina** con las figuras que lo sustentan (9.3), la **matriz de alcance** (9.4) y la **lista de ajustes** (9.6). Cada hallazgo debe tener una cifra que salga del notebook, no una impresión.

In [ ]:
# 9.1 Tabla de hallazgos 🔧
# Saque las cifras de las tablas del notebook (vea 8.2). Ejemplo de cómo tomar un valor:
#   f"{val_ppt['KGE'].median():.2f}"  ->  KGE mediano de CHIRPS frente a las estaciones
HALLAZGOS = [
    # (pregunta o componente, resultado con su cifra, tabla o figura de donde sale, qué implica para la propuesta)
]

if not HALLAZGOS:
    print("⚠️ Complete HALLAZGOS con lo que encontró su grupo y vuelva a ejecutar esta celda.")
else:
    hallazgos = pd.DataFrame(HALLAZGOS, columns=["pregunta_o_componente", "resultado", "fuente_en_el_notebook", "implicacion"]).assign(grupo=GRUPO)
    guardar_tabla(hallazgos, f"9_hallazgos_{sufijo}")
    display(Markdown(f"### Hallazgos · Grupo {GRUPO}\n\n" + "\n".join(
        f"- **{r.pregunta_o_componente}:** {r.resultado} ({r.fuente_en_el_notebook}). {r.implicacion}" for r in hallazgos.itertuples())))

In [ ]:
# 9.2 Figura propia 🔧
# Construya la figura que mejor muestre su hallazgo principal, con las tablas del notebook o con sus datos.
# Ideas: combinar dos tablas (p. ej. porcentaje de días con dato y altura), filtrar las estaciones o subcuencas de su área,
# comparar dos configuraciones que usted ejecutó.
FIGURA_PROPIA = False   # cámbielo a True cuando escriba su código

if FIGURA_PROPIA:
    fig, ax = plt.subplots(figsize=(9, 5))
    # --- su código aquí ---

    guardar_fig(fig, f"9_figura_{sufijo}")
    plt.show()
else:
    print("⚠️ Escriba su figura en esta celda y ponga FIGURA_PROPIA = True.")

In [ ]:
# 9.3 Lámina de hallazgos 🔧
# Elija las figuras que sustentan sus hallazgos (nombres de la lista 8.2, sin la extensión .png),
# en el orden en que las va a presentar. Incluya su figura propia: f"9_figura_{sufijo}"
LAMINA = [
    # "2_mapa_datos",
]

import matplotlib.image as mpimg
_paneles = [n for n in LAMINA if os.path.exists(os.path.join(CARPETA_SALIDA, f"{n}.png"))]
_faltan = sorted(set(LAMINA) - set(_paneles))
if _faltan:
    print("No encontré estas figuras (revise los nombres en 8.2):", ", ".join(_faltan))
if not _paneles:
    print("⚠️ Agregue a LAMINA las figuras que quiere presentar y vuelva a ejecutar esta celda.")
else:
    _f = (len(_paneles) + 1) // 2
    fig, axes = plt.subplots(_f, 2 if len(_paneles) > 1 else 1, figsize=(16 if len(_paneles) > 1 else 9, 5.5 * _f), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, n in zip(axes.flat, _paneles):
        ax.imshow(mpimg.imread(os.path.join(CARPETA_SALIDA, f"{n}.png")))
        ax.set_title(n, fontsize=11, loc="left")
    fig.suptitle(f"Hallazgos · Grupo {GRUPO}", fontsize=15)
    plt.tight_layout()
    guardar_fig(fig, f"9_lamina_{sufijo}")
    plt.show()

### 9.4 Matriz de alcance

Una fila por variable o indicador de **su propuesta**, con la evidencia que sacó del notebook.

- **Componente o etapa:** peligro, exposición, vulnerabilidad, o la etapa de su metodología.
- **Disponibilidad y cobertura:** qué tan completa está la fuente para su periodo y su área (con la cifra).
- **Esfuerzo:** días de trabajo estimados para obtener y procesar la variable.
- **Decisión:** Mantener, Sustituir (por otra fuente ya publicada) o Recortar.

In [ ]:
# 9.4 Matriz de alcance 🔧
MATRIZ = [
    # (variable, componente o etapa, fuente, disponibilidad y cobertura, esfuerzo en días, ¿responde la pregunta?, decisión, justificación)
]

cols_m = ["variable", "componente_o_etapa", "fuente", "disponibilidad_y_cobertura", "esfuerzo_dias",
          "responde_la_pregunta", "decision", "justificacion"]
if not MATRIZ:
    print("⚠️ Complete MATRIZ con las variables de su propuesta y vuelva a ejecutar esta celda.")
else:
    matriz = pd.DataFrame(MATRIZ, columns=cols_m)
    guardar_tabla(matriz, f"9_matriz_alcance_{sufijo}")
    matriz.to_excel(os.path.join(CARPETA_SALIDA, f"9_matriz_alcance_{sufijo}.xlsx"), index=False)
    print("Esfuerzo total estimado:", pd.to_numeric(matriz["esfuerzo_dias"], errors="coerce").sum(), "días")
    display(matriz)

### 9.5 Nota de ajustes (una página)

Haga doble clic en esta celda y complétela con base en sus hallazgos (9.1) y su matriz (9.4).

**Grupo:** _ · **Propuesta:** _

1. **Qué encontramos al ejecutar el notebook:** …
2. **Qué mantenemos y por qué:** …
3. **Qué sustituimos (y por cuál fuente):** …
4. **Qué recortamos:** …
5. **Cómo respondemos al comentario de la evaluación:** …
6. **Riesgos que persisten y cómo los mitigamos:** …

### 9.6 Lista de ajustes

Resuma en tres a cinco líneas qué recorta, qué sustituye y por qué. Junto con la lámina, es lo que el grupo presenta.

In [ ]:
# 9.6 Lista de ajustes 🔧
AJUSTES = [
    # (decisión: Mantener, Sustituir o Recortar · qué · por qué)
]

if not AJUSTES:
    print("⚠️ Complete AJUSTES y vuelva a ejecutar esta celda.")
else:
    ajustes = pd.DataFrame(AJUSTES, columns=["decision", "que", "por_que"]).assign(grupo=GRUPO)
    guardar_tabla(ajustes, f"9_lista_ajustes_{sufijo}")
    display(Markdown(f"**Grupo {GRUPO} · ajustes a la propuesta**\n\n" +
                     "\n".join(f"- **{r.decision}:** {r.que}. {r.por_que}" for r in ajustes.itertuples())))

In [ ]:
# 9.7 Descargar todos los resultados (tablas, figuras, mapas y lo que construyó el grupo) en un .zip
ruta_zip = os.path.join(os.path.dirname(CARPETA_SALIDA.rstrip("/")) or ".", f"Escuela-ITS2026_resultados_{sufijo}.zip")
with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(CARPETA_SALIDA, "*"))):
        z.write(f, os.path.basename(f))
print(f"{len(glob.glob(os.path.join(CARPETA_SALIDA, '*')))} archivos en {ruta_zip}")
if EN_COLAB:
    from google.colab import files
    files.download(ruta_zip)